# MyoLab-AI — Day 35: Confidence Calibration & Abstention

Notebook standalone cho hai profile:

- `mendeley`
- `grabmyo`

Chuỗi contract:

```text
ETL canonical NPZ
→ Day 32 frozen baseline
→ Day 34 frozen personalization policy
→ Day 35 calibration development
→ frozen abstention policy
→ one-time validation evaluation
```

## Mục tiêu

1. Hiệu chỉnh xác suất cho cả `P0` và `P1`.
2. Không thay đổi predicted class khi calibration.
3. Tách ba nhóm training subjects độc lập:
   - `calibration_fit`;
   - `calibrator_select`;
   - `threshold_select`.
4. Không dùng validation subjects để fit calibrator, chọn calibrator, chọn confidence score hoặc chọn threshold.
5. Xuất reliability, ECE, NLL, Brier, coverage–risk, AURC và selective metrics.
6. Đóng gói bundle dùng tiếp cho Day 36.

> Đây là đánh giá kỹ thuật offline. Không phải chẩn đoán, không phải chỉ định điều trị và chưa được phê duyệt cho sử dụng lâm sàng.

## Bước 1 — Môi trường, governance và cấu hình

**Input:** lựa chọn dataset profile và các artifact đã PASS của Day 34.  
**Output:** đường dẫn runtime/persistent cùng contract Day 35 đã khóa.

In [14]:
# === CELL 1: Environment, governance, dataset profile and paths ===
from __future__ import annotations

import hashlib
import json
import math
import os
import platform
import shutil
import time
import warnings
import zipfile
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import sklearn
from scipy.optimize import minimize_scalar

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    precision_score,
    recall_score,
)

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive, files  # type: ignore
    if not Path('/content/drive/MyDrive').exists():
        drive.mount('/content/drive')

# -----------------------------------------------------------------------------
# Governance
# -----------------------------------------------------------------------------
DAY35_CONFIDENCE_CALIBRATION_AUTHORIZED = True
BASELINE_REFIT_ALLOWED = False
PERSONALIZATION_POLICY_REFIT_ALLOWED = False
VALIDATION_CALIBRATOR_FIT_ALLOWED = False
VALIDATION_CALIBRATOR_SELECTION_ALLOWED = False
VALIDATION_THRESHOLD_SELECTION_ALLOWED = False
TEST_SET_OPENED = False
POOLED_DATASET_ALLOWED = False
CLINICAL_USE_ALLOWED = False

assert DAY35_CONFIDENCE_CALIBRATION_AUTHORIZED is True
assert BASELINE_REFIT_ALLOWED is False
assert PERSONALIZATION_POLICY_REFIT_ALLOWED is False
assert VALIDATION_CALIBRATOR_FIT_ALLOWED is False
assert VALIDATION_CALIBRATOR_SELECTION_ALLOWED is False
assert VALIDATION_THRESHOLD_SELECTION_ALLOWED is False
assert TEST_SET_OPENED is False
assert POOLED_DATASET_ALLOWED is False
assert CLINICAL_USE_ALLOWED is False

# -----------------------------------------------------------------------------
# Run configuration
# -----------------------------------------------------------------------------
DATASET_PROFILE = os.environ.get('DAY35_DATASET_PROFILE', 'grabmyo').strip().lower()
if DATASET_PROFILE not in {'mendeley', 'grabmyo'}:
    raise ValueError("DAY35_DATASET_PROFILE phải là 'mendeley' hoặc 'grabmyo'.")

RUN_ID = f'day35-confidence-{DATASET_PROFILE}-v2'
DAY34_RUN_ID = f'day34-personalization-{DATASET_PROFILE}-v2'

K_VALUES = (2, 3)
ARMS = ('p0', 'p1')
DAY35_EPISODE_SEED = 3501
RANDOM_SEED = 3501
CHUNK_SIZE = 25000
N_CALIBRATION_BINS = 15
MIN_TEMPERATURE_NLL_GAIN = 1e-4
OPERATING_COVERAGE_TARGETS = (0.95, 0.90, 0.80)
PRIMARY_OPERATING_POINT = 'coverage_90'
BOOTSTRAP_RESAMPLES = 3000
QUICK_SMOKE_TEST = os.environ.get('DAY35_QUICK_SMOKE', '0') == '1'
DOWNLOAD_HANDOFF_TO_BROWSER = False

if QUICK_SMOKE_TEST:
    BOOTSTRAP_RESAMPLES = 200

DRIVE_DATA_ROOT = Path('/content/drive/MyDrive/MyoLab-AI-data')

PROFILE_CONFIG = {
    'mendeley': {
        'dataset_slug': 'mendeley-4channel-hand-gesture-v2',
        'expected_train_subjects': 32,
        'expected_validation_subjects': 8,
        'expected_classes': 4,
        'p0_aggregation': 'window_vote_fraction',
        'calibration_pool': 'all_trials',
        'primary_scope': 'all_eval',
    },
    'grabmyo': {
        'dataset_slug': 'grabmyo-physionet-v1.1.0',
        'expected_train_subjects': 34,
        'expected_validation_subjects': 9,
        'expected_classes': 4,
        'p0_aggregation': 'mean_window_model_score',
        'calibration_pool': 'session1_only',
        'primary_scope': 'cross_session',
    },
}
PROFILE = PROFILE_CONFIG[DATASET_PROFILE]
DATASET_ROOT = DRIVE_DATA_ROOT / PROFILE['dataset_slug']
DAY34_ROOT = DATASET_ROOT / 'outputs' / 'day34-personalization' / DAY34_RUN_ID

if IN_COLAB:
    RUNTIME_ROOT = Path('/content/data') / PROFILE['dataset_slug'] / 'day35-confidence' / RUN_ID
else:
    RUNTIME_ROOT = Path.cwd() / '.day35-runtime' / PROFILE['dataset_slug'] / RUN_ID

PERSIST_ROOT = DATASET_ROOT / 'outputs' / 'day35-confidence' / RUN_ID
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)

np.random.seed(RANDOM_SEED)

print('=' * 96)
print('[CELL 1] DAY 35 CONFIGURATION')
print('=' * 96)
print(f'Dataset profile                : {DATASET_PROFILE}')
print(f'Dataset root                   : {DATASET_ROOT}')
print(f'Day 34 root                    : {DAY34_ROOT}')
print(f'Runtime output                 : {RUNTIME_ROOT}')
print(f'Persistent output              : {PERSIST_ROOT}')
print(f'k values                       : {K_VALUES}')
print(f'Arms                           : {ARMS}')
print(f'Development episode seed       : {DAY35_EPISODE_SEED}')
print(f'Operating coverage targets     : {OPERATING_COVERAGE_TARGETS}')
print(f'Primary operating point        : {PRIMARY_OPERATING_POINT}')
print(f'Quick smoke test               : {QUICK_SMOKE_TEST}')
print(f'Baseline refit allowed         : {BASELINE_REFIT_ALLOWED}')
print(f'Validation threshold tuning    : {VALIDATION_THRESHOLD_SELECTION_ALLOWED}')
print(f'Clinical use allowed           : {CLINICAL_USE_ALLOWED}')
print('[PASS] Governance and configuration initialized.')

[CELL 1] DAY 35 CONFIGURATION
Dataset profile                : grabmyo
Dataset root                   : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0
Day 34 root                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2
Runtime output                 : /content/data/grabmyo-physionet-v1.1.0/day35-confidence/day35-confidence-grabmyo-v2
Persistent output              : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day35-confidence/day35-confidence-grabmyo-v2
k values                       : (2, 3)
Arms                           : ('p0', 'p1')
Development episode seed       : 3501
Operating coverage targets     : (0.95, 0.9, 0.8)
Primary operating point        : coverage_90
Quick smoke test               : False
Baseline refit allowed         : False
Validation threshold tuning    : False
Clinical use allowed           : False
[PASS] Governance and configurati

## Bước 2 — Helpers cho calibration, coverage–risk và artifact

**Input:** xác suất multiclass, nhãn thật, subject IDs.  
**Output:** calibrator, calibration metrics, confidence scores, coverage–risk curve và các hàm ghi artifact an toàn.

In [15]:
# === CELL 2: Common helpers, calibration metrics and selective prediction ===
def utc_now_iso() -> str:
    return datetime.now(timezone.utc).isoformat()


def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file_obj:
        while True:
            chunk = file_obj.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(key): json_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [json_safe(item) for item in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, float) and not math.isfinite(value):
        return None
    return value


def write_json_atomic(payload: dict[str, Any], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    temporary.write_text(
        json.dumps(json_safe(payload), ensure_ascii=False, indent=2),
        encoding='utf-8',
    )
    temporary.replace(path)


def write_dataframe_atomic(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    compression = 'gzip' if path.suffix == '.gz' else None
    frame.to_csv(temporary, index=False, compression=compression)
    temporary.replace(path)


def persist_artifact(runtime_path: Path) -> Path:
    persistent_path = PERSIST_ROOT / runtime_path.name
    temporary = persistent_path.with_suffix(persistent_path.suffix + '.part')
    shutil.copy2(runtime_path, temporary)
    temporary.replace(persistent_path)
    return persistent_path


def resolve_artifact(
    *,
    override_env: str,
    exact_path: Path,
    search_root: Path,
    filename: str,
) -> Path:
    override = os.environ.get(override_env, '').strip()
    candidates = [Path(override)] if override else []
    candidates.append(exact_path)
    for candidate in candidates:
        if candidate.exists():
            return candidate
    matches = sorted(search_root.rglob(filename)) if search_root.exists() else []
    if len(matches) == 1:
        return matches[0]
    if not matches:
        raise FileNotFoundError(
            f'Không tìm thấy {filename}. Exact path: {exact_path}. '
            f'Có thể đặt {override_env}.'
        )
    raise RuntimeError(
        f'Tìm thấy nhiều artifact tên {filename}; cần đặt {override_env}:\n'
        + '\n'.join(str(path) for path in matches)
    )


def stable_seed(*parts: Any) -> int:
    text = '||'.join(str(part) for part in parts)
    digest = hashlib.sha256(text.encode('utf-8')).digest()
    return int.from_bytes(digest[:8], 'little') % (2**32 - 1)


def scalar_text(array: np.ndarray) -> str:
    return str(np.asarray(array).reshape(-1)[0])


def softmax_rows(values: np.ndarray) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    shifted = values - np.max(values, axis=1, keepdims=True)
    exp_values = np.exp(shifted)
    denominator = exp_values.sum(axis=1, keepdims=True)
    if np.any(denominator <= 0) or not np.isfinite(denominator).all():
        raise RuntimeError('Softmax denominator không hợp lệ.')
    return exp_values / denominator


def normalize_probabilities(probabilities: np.ndarray) -> np.ndarray:
    values = np.asarray(probabilities, dtype=np.float64)
    if values.ndim != 2:
        raise ValueError(f'Probability matrix phải là 2D, nhận {values.shape}.')
    values = np.clip(values, 0.0, None)
    denominator = values.sum(axis=1, keepdims=True)
    if np.any(denominator <= 0):
        raise RuntimeError('Probability row có tổng bằng 0.')
    values = values / denominator
    if not np.isfinite(values).all():
        raise RuntimeError('Probability matrix chứa NaN/Inf.')
    return values


def estimator_classes(estimator: Any) -> np.ndarray:
    if hasattr(estimator, 'classes_'):
        return np.asarray(estimator.classes_).astype(str)
    if hasattr(estimator, 'named_steps'):
        for step in reversed(list(estimator.named_steps.values())):
            if hasattr(step, 'classes_'):
                return np.asarray(step.classes_).astype(str)
    raise RuntimeError('Không tìm thấy classes_ trong frozen baseline model.')


def aligned_score_matrix(
    estimator: Any,
    X_input: np.ndarray,
    class_order: np.ndarray,
) -> tuple[np.ndarray, str]:
    if hasattr(estimator, 'predict_proba'):
        raw = np.asarray(estimator.predict_proba(X_input), dtype=np.float64)
        score_kind = 'predict_proba'
    elif hasattr(estimator, 'decision_function'):
        raw = np.asarray(estimator.decision_function(X_input), dtype=np.float64)
        if raw.ndim == 1:
            raw = np.column_stack([-raw, raw])
        score_kind = 'decision_function'
    else:
        predictions = np.asarray(estimator.predict(X_input)).astype(str)
        raw = np.zeros((len(predictions), len(class_order)), dtype=np.float64)
        for class_index, label in enumerate(class_order):
            raw[:, class_index] = predictions == label
        return raw, 'one_hot_prediction'

    classes = estimator_classes(estimator)
    aligned = np.full((raw.shape[0], len(class_order)), np.nan, dtype=np.float64)
    for source_index, label in enumerate(classes):
        matches = np.flatnonzero(class_order == label)
        if len(matches) != 1:
            raise RuntimeError(f'Estimator class không thuộc frozen class order: {label}')
        aligned[:, int(matches[0])] = raw[:, source_index]
    if not np.isfinite(aligned).all():
        raise RuntimeError('Không align được score matrix với class_order.')
    return aligned, score_kind


def normalize_global_trial_scores(values: np.ndarray, score_kind: str) -> np.ndarray:
    values = np.asarray(values, dtype=np.float64)
    if score_kind in {'predict_proba', 'one_hot_prediction', 'window_vote_fraction'}:
        return normalize_probabilities(values)
    return softmax_rows(values)


def labels_to_indices(labels: Iterable[str], class_order: np.ndarray) -> np.ndarray:
    mapping = {str(label): index for index, label in enumerate(class_order)}
    result = np.asarray([mapping[str(label)] for label in labels], dtype=np.int64)
    return result


def subject_equal_weights(subject_ids: Iterable[str]) -> np.ndarray:
    subjects = np.asarray(list(subject_ids)).astype(str)
    counts = pd.Series(subjects).value_counts().to_dict()
    weights = np.asarray([1.0 / counts[subject] for subject in subjects], dtype=np.float64)
    weights = weights / weights.sum() * len(weights)
    return weights


def weighted_mean(values: np.ndarray, weights: np.ndarray | None = None) -> float:
    values = np.asarray(values, dtype=np.float64)
    if weights is None:
        return float(np.mean(values))
    weights = np.asarray(weights, dtype=np.float64)
    return float(np.sum(values * weights) / np.sum(weights))


def multiclass_nll(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    weights: np.ndarray | None = None,
) -> float:
    probabilities = normalize_probabilities(probabilities)
    chosen = probabilities[np.arange(len(y_index)), np.asarray(y_index, dtype=np.int64)]
    losses = -np.log(np.clip(chosen, 1e-12, 1.0))
    return weighted_mean(losses, weights)


def multiclass_brier(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    weights: np.ndarray | None = None,
) -> float:
    probabilities = normalize_probabilities(probabilities)
    one_hot = np.zeros_like(probabilities)
    one_hot[np.arange(len(y_index)), np.asarray(y_index, dtype=np.int64)] = 1.0
    per_row = np.sum((probabilities - one_hot) ** 2, axis=1)
    return weighted_mean(per_row, weights)


def calibration_bin_table(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 15,
    strategy: str = 'equal_width',
    weights: np.ndarray | None = None,
) -> pd.DataFrame:
    probabilities = normalize_probabilities(probabilities)
    y_index = np.asarray(y_index, dtype=np.int64)
    confidence = np.max(probabilities, axis=1)
    prediction = np.argmax(probabilities, axis=1)
    correct = prediction == y_index
    if weights is None:
        weights = np.ones(len(y_index), dtype=np.float64)
    else:
        weights = np.asarray(weights, dtype=np.float64)

    if strategy == 'equal_width':
        edges = np.linspace(0.0, 1.0, n_bins + 1)
        assignments = np.clip(np.digitize(confidence, edges[1:-1], right=True), 0, n_bins - 1)
    elif strategy == 'equal_mass':
        order = np.argsort(confidence)
        assignments = np.empty(len(confidence), dtype=np.int64)
        chunks = np.array_split(order, min(n_bins, len(order)))
        for bin_index, indices in enumerate(chunks):
            assignments[indices] = bin_index
        edges = None
    else:
        raise ValueError(f'Unknown calibration bin strategy: {strategy}')

    rows = []
    total_weight = float(weights.sum())
    for bin_index in sorted(np.unique(assignments).tolist()):
        mask = assignments == bin_index
        bin_weight = float(weights[mask].sum())
        if bin_weight <= 0:
            continue
        rows.append({
            'bin_index': int(bin_index),
            'strategy': strategy,
            'n_rows': int(mask.sum()),
            'weight_fraction': bin_weight / total_weight,
            'confidence_mean': weighted_mean(confidence[mask], weights[mask]),
            'accuracy_mean': weighted_mean(correct[mask].astype(float), weights[mask]),
            'confidence_min': float(confidence[mask].min()),
            'confidence_max': float(confidence[mask].max()),
        })
    return pd.DataFrame(rows)


def expected_calibration_error(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    n_bins: int = 15,
    strategy: str = 'equal_width',
    weights: np.ndarray | None = None,
) -> float:
    table = calibration_bin_table(
        y_index=y_index,
        probabilities=probabilities,
        n_bins=n_bins,
        strategy=strategy,
        weights=weights,
    )
    return float(np.sum(
        table['weight_fraction'].to_numpy(dtype=float)
        * np.abs(
            table['accuracy_mean'].to_numpy(dtype=float)
            - table['confidence_mean'].to_numpy(dtype=float)
        )
    ))


def calibration_metrics(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    weights: np.ndarray | None = None,
) -> dict[str, float]:
    probabilities = normalize_probabilities(probabilities)
    prediction = np.argmax(probabilities, axis=1)
    return {
        'nll': multiclass_nll(y_index, probabilities, weights),
        'brier': multiclass_brier(y_index, probabilities, weights),
        'ece_equal_width': expected_calibration_error(
            y_index, probabilities, N_CALIBRATION_BINS, 'equal_width', weights
        ),
        'ece_equal_mass': expected_calibration_error(
            y_index, probabilities, N_CALIBRATION_BINS, 'equal_mass', weights
        ),
        'accuracy': weighted_mean((prediction == y_index).astype(float), weights),
        'confidence_mean': weighted_mean(np.max(probabilities, axis=1), weights),
    }


@dataclass
class IdentityCalibrator:
    method: str = 'identity'

    def fit(
        self,
        probabilities: np.ndarray,
        y_index: np.ndarray,
        sample_weight: np.ndarray | None = None,
    ) -> 'IdentityCalibrator':
        normalize_probabilities(probabilities)
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        return normalize_probabilities(probabilities)

    def metadata(self) -> dict[str, Any]:
        return {'method': self.method}


@dataclass
class ScalarTemperatureCalibrator:
    temperature: float = 1.0
    method: str = 'scalar_temperature'

    def fit(
        self,
        probabilities: np.ndarray,
        y_index: np.ndarray,
        sample_weight: np.ndarray | None = None,
    ) -> 'ScalarTemperatureCalibrator':
        base_probabilities = normalize_probabilities(probabilities)
        log_probabilities = np.log(np.clip(base_probabilities, 1e-12, 1.0))
        y_index = np.asarray(y_index, dtype=np.int64)

        def objective(log_temperature: float) -> float:
            temperature = float(np.exp(log_temperature))
            calibrated = softmax_rows(log_probabilities / temperature)
            return multiclass_nll(y_index, calibrated, sample_weight)

        result = minimize_scalar(
            objective,
            bounds=(math.log(0.05), math.log(20.0)),
            method='bounded',
            options={'xatol': 1e-7, 'maxiter': 500},
        )
        if not result.success:
            raise RuntimeError(f'Temperature optimization failed: {result.message}')
        self.temperature = float(np.exp(result.x))
        return self

    def predict_proba(self, probabilities: np.ndarray) -> np.ndarray:
        base_probabilities = normalize_probabilities(probabilities)
        log_probabilities = np.log(np.clip(base_probabilities, 1e-12, 1.0))
        return softmax_rows(log_probabilities / float(self.temperature))

    def metadata(self) -> dict[str, Any]:
        return {'method': self.method, 'temperature': float(self.temperature)}


def confidence_scores(probabilities: np.ndarray) -> dict[str, np.ndarray]:
    probabilities = normalize_probabilities(probabilities)
    ordered = np.sort(probabilities, axis=1)
    entropy = -np.sum(
        probabilities * np.log(np.clip(probabilities, 1e-12, 1.0)),
        axis=1,
    )
    normalized_entropy = entropy / math.log(probabilities.shape[1])
    return {
        'max_probability': ordered[:, -1],
        'probability_margin': ordered[:, -1] - ordered[:, -2],
        'one_minus_normalized_entropy': 1.0 - normalized_entropy,
    }


def exact_aurc(y_index: np.ndarray, probabilities: np.ndarray, score: np.ndarray) -> float:
    y_index = np.asarray(y_index, dtype=np.int64)
    probabilities = normalize_probabilities(probabilities)
    score = np.asarray(score, dtype=np.float64)
    prediction = np.argmax(probabilities, axis=1)
    errors = (prediction != y_index).astype(np.float64)
    order = np.argsort(-score, kind='mergesort')
    cumulative_risk = np.cumsum(errors[order]) / np.arange(1, len(errors) + 1)
    return float(np.mean(cumulative_risk))


def risk_coverage_curve(
    y_index: np.ndarray,
    probabilities: np.ndarray,
    score: np.ndarray,
    n_points: int = 101,
) -> pd.DataFrame:
    y_index = np.asarray(y_index, dtype=np.int64)
    probabilities = normalize_probabilities(probabilities)
    score = np.asarray(score, dtype=np.float64)
    prediction = np.argmax(probabilities, axis=1)
    correct = prediction == y_index

    quantiles = np.linspace(0.0, 1.0, n_points)
    thresholds = np.unique(np.quantile(score, quantiles))
    rows = []
    for threshold in thresholds:
        accepted = score >= float(threshold)
        accepted_count = int(accepted.sum())
        coverage = accepted_count / len(score)
        if accepted_count:
            selective_risk = float(np.mean(~correct[accepted]))
            selective_accuracy = float(np.mean(correct[accepted]))
        else:
            selective_risk = float('nan')
            selective_accuracy = float('nan')
        rows.append({
            'threshold': float(threshold),
            'coverage': float(coverage),
            'selective_risk': selective_risk,
            'selective_accuracy': selective_accuracy,
            'accepted_count': accepted_count,
            'abstained_count': int(len(score) - accepted_count),
        })
    return pd.DataFrame(rows).sort_values('coverage', ascending=False).reset_index(drop=True)


def quantile_threshold_for_coverage(score: np.ndarray, target_coverage: float) -> float:
    if not 0 < target_coverage <= 1:
        raise ValueError('target_coverage phải nằm trong (0, 1].')
    score = np.asarray(score, dtype=np.float64)
    sorted_score = np.sort(score)
    index = max(0, int(math.floor((1.0 - target_coverage) * len(sorted_score))))
    index = min(index, len(sorted_score) - 1)
    return float(sorted_score[index])


def classification_metrics(
    y_true: Iterable[str],
    y_pred: Iterable[str],
    labels: np.ndarray,
) -> dict[str, float]:
    y_true = np.asarray(list(y_true)).astype(str)
    y_pred = np.asarray(list(y_pred)).astype(str)
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'macro_f1': float(f1_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)),
        'macro_precision': float(
            precision_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
        ),
        'macro_recall': float(
            recall_score(y_true, y_pred, labels=labels, average='macro', zero_division=0)
        ),
    }


def selective_metrics_from_frame(
    frame: pd.DataFrame,
    *,
    accepted_column: str,
    truth_column: str,
    prediction_column: str,
    labels: np.ndarray,
) -> dict[str, Any]:
    accepted = frame[accepted_column].astype(bool).to_numpy()
    correct = (
        frame[prediction_column].astype(str).to_numpy()
        == frame[truth_column].astype(str).to_numpy()
    )
    accepted_count = int(accepted.sum())
    total_count = int(len(frame))
    total_errors = int((~correct).sum())
    accepted_errors = int((~correct & accepted).sum())
    abstained_errors = int((~correct & ~accepted).sum())

    result: dict[str, Any] = {
        'n_rows': total_count,
        'accepted_count': accepted_count,
        'abstained_count': total_count - accepted_count,
        'coverage': float(accepted_count / total_count) if total_count else float('nan'),
        'abstention_rate': float(1.0 - accepted_count / total_count) if total_count else float('nan'),
        'selective_risk': float(accepted_errors / accepted_count) if accepted_count else float('nan'),
        'selective_accuracy': float(1.0 - accepted_errors / accepted_count) if accepted_count else float('nan'),
        'total_error_count': total_errors,
        'accepted_error_count': accepted_errors,
        'abstained_error_count': abstained_errors,
        'error_capture_rate': float(abstained_errors / total_errors) if total_errors else 0.0,
    }
    if accepted_count:
        accepted_frame = frame.loc[accepted]
        result.update({
            f'selective_{name}': value
            for name, value in classification_metrics(
                accepted_frame[truth_column],
                accepted_frame[prediction_column],
                labels,
            ).items()
        })
    else:
        for name in ('accuracy', 'balanced_accuracy', 'macro_f1', 'macro_precision', 'macro_recall'):
            result[f'selective_{name}'] = float('nan')
    return result


def bootstrap_subject_mean_ci(
    subject_frame: pd.DataFrame,
    value_column: str,
    seed: int,
    n_resamples: int,
) -> tuple[float, float]:
    values = subject_frame[value_column].to_numpy(dtype=np.float64)
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return float('nan'), float('nan')
    rng = np.random.default_rng(seed)
    sample_indices = rng.integers(0, len(values), size=(n_resamples, len(values)))
    means = values[sample_indices].mean(axis=1)
    return float(np.quantile(means, 0.025)), float(np.quantile(means, 0.975))


print('[PASS] Calibration and selective-prediction helpers initialized.')

[PASS] Calibration and selective-prediction helpers initialized.


## Bước 3 — Locate và audit Day 34 artifacts

**Input:** Day 34 handoff của đúng dataset profile.  
**Output:** input gate xác nhận Day 34 PASS, checksum khớp, score columns hợp lệ và validation chưa được dùng cho Day 35 development.

In [16]:
# === CELL 3: Locate and validate Day 34 artifacts ===
DAY34_GATE_PATH = resolve_artifact(
    override_env='DAY35_DAY34_GATE_PATH',
    exact_path=DAY34_ROOT / 'day34-final-gate.json',
    search_root=DATASET_ROOT,
    filename='day34-final-gate.json',
)
DAY34_EVIDENCE_PATH = resolve_artifact(
    override_env='DAY35_DAY34_EVIDENCE_PATH',
    exact_path=DAY34_ROOT / 'day34-personalization-evidence.json',
    search_root=DATASET_ROOT,
    filename='day34-personalization-evidence.json',
)
DAY34_INPUT_GATE_PATH = resolve_artifact(
    override_env='DAY35_DAY34_INPUT_GATE_PATH',
    exact_path=DAY34_ROOT / 'day34-input-gate.json',
    search_root=DATASET_ROOT,
    filename='day34-input-gate.json',
)
DAY34_PROTOCOL_PATH = resolve_artifact(
    override_env='DAY35_DAY34_PROTOCOL_PATH',
    exact_path=DAY34_ROOT / 'few-shot-protocol.json',
    search_root=DATASET_ROOT,
    filename='few-shot-protocol.json',
)
DAY34_POLICY_SELECTION_PATH = resolve_artifact(
    override_env='DAY35_DAY34_POLICY_SELECTION_PATH',
    exact_path=DAY34_ROOT / 'few-shot-policy-selection.json',
    search_root=DATASET_ROOT,
    filename='few-shot-policy-selection.json',
)
DAY34_ADAPTER_BUNDLE_PATH = resolve_artifact(
    override_env='DAY35_DAY34_ADAPTER_BUNDLE_PATH',
    exact_path=DAY34_ROOT / 'few-shot-adapter-bundle.joblib',
    search_root=DATASET_ROOT,
    filename='few-shot-adapter-bundle.joblib',
)
DAY34_VALIDATION_PREDICTIONS_PATH = resolve_artifact(
    override_env='DAY35_DAY34_VALIDATION_PREDICTIONS_PATH',
    exact_path=DAY34_ROOT / 'few-shot-trial-predictions.csv.gz',
    search_root=DATASET_ROOT,
    filename='few-shot-trial-predictions.csv.gz',
)
DAY34_CALIBRATION_MANIFEST_PATH = resolve_artifact(
    override_env='DAY35_DAY34_CALIBRATION_MANIFEST_PATH',
    exact_path=DAY34_ROOT / 'few-shot-calibration-manifest.csv',
    search_root=DATASET_ROOT,
    filename='few-shot-calibration-manifest.csv',
)

day34_gate = json.loads(DAY34_GATE_PATH.read_text(encoding='utf-8'))
day34_evidence = json.loads(DAY34_EVIDENCE_PATH.read_text(encoding='utf-8'))
day34_input_gate = json.loads(DAY34_INPUT_GATE_PATH.read_text(encoding='utf-8'))
day34_protocol = json.loads(DAY34_PROTOCOL_PATH.read_text(encoding='utf-8'))
day34_policy_selection = json.loads(DAY34_POLICY_SELECTION_PATH.read_text(encoding='utf-8'))
adapter_bundle = joblib.load(DAY34_ADAPTER_BUNDLE_PATH)

if not str(day34_gate.get('status', '')).startswith('PASS'):
    raise RuntimeError(f'Day 34 gate chưa PASS: {day34_gate.get("status")}')
if day34_protocol.get('dataset_profile') != DATASET_PROFILE:
    raise RuntimeError('Day 34 protocol dataset profile không khớp Day 35.')
if adapter_bundle.get('dataset_profile') != DATASET_PROFILE:
    raise RuntimeError('Adapter bundle dataset profile không khớp.')
if day34_protocol.get('validation_policy_tuning_allowed') is not False:
    raise RuntimeError('Day 34 protocol không khóa validation tuning.')
if day34_protocol.get('baseline_refit_allowed') is not False:
    raise RuntimeError('Day 34 protocol không khóa baseline refit.')

class_order = np.asarray(day34_protocol['class_order']).astype(str)
if len(class_order) != PROFILE['expected_classes']:
    raise RuntimeError(f'Expected {PROFILE["expected_classes"]} classes, got {class_order.tolist()}.')

validation_predictions_df = pd.read_csv(
    DAY34_VALIDATION_PREDICTIONS_PATH,
    compression='infer',
    dtype={
        'subject_id': str,
        'trial_key': str,
        'record_id': str,
        'repetition_id': str,
        'episode_id': str,
        'split_name': str,
        'y_true': str,
    },
)
day34_calibration_manifest_df = pd.read_csv(
    DAY34_CALIBRATION_MANIFEST_PATH,
    dtype={
        'subject_id': str,
        'trial_key': str,
        'record_id': str,
        'repetition_id': str,
        'episode_id': str,
        'split_name': str,
        'class_label': str,
    },
)

required_prediction_columns = {
    'episode_id', 'subject_id', 'trial_key', 'record_id', 'repetition_id',
    'session_id', 'split_name', 'y_true', 'k', 'seed', 'eval_scope',
    'p0_pred', 'p1_pred',
}
for arm in ARMS:
    required_prediction_columns.update({f'{arm}_prob__{label}' for label in class_order})
missing_prediction_columns = required_prediction_columns - set(validation_predictions_df.columns)
if missing_prediction_columns:
    raise RuntimeError(
        f'Day 34 validation predictions thiếu columns: {sorted(missing_prediction_columns)}'
    )

if set(validation_predictions_df['split_name'].astype(str)) != {'validation'}:
    raise RuntimeError(
        f'Day 34 validation prediction splits không hợp lệ: '
        f'{sorted(validation_predictions_df["split_name"].unique().tolist())}'
    )
if set(validation_predictions_df['y_true'].astype(str)) != set(class_order):
    raise RuntimeError('Day 34 validation labels không khớp class_order.')
if set(validation_predictions_df['k'].astype(int).unique()) != set(K_VALUES):
    raise RuntimeError('Day 34 validation predictions không có đủ k=2 và k=3.')

for arm in ARMS:
    columns = [f'{arm}_prob__{label}' for label in class_order]
    matrix = validation_predictions_df[columns].to_numpy(dtype=np.float64)
    if not np.isfinite(matrix).all():
        raise RuntimeError(f'{arm} validation probabilities chứa NaN/Inf.')
    matrix = normalize_probabilities(matrix)
    if not np.allclose(matrix.sum(axis=1), 1.0, atol=1e-8):
        raise RuntimeError(f'{arm} validation probability rows không sum về 1.')

manifest_pairs = set(zip(
    day34_calibration_manifest_df['episode_id'].astype(str),
    day34_calibration_manifest_df['trial_key'].astype(str),
))
evaluation_pairs = set(zip(
    validation_predictions_df['episode_id'].astype(str),
    validation_predictions_df['trial_key'].astype(str),
))
day34_overlap_count = len(manifest_pairs & evaluation_pairs)
if day34_overlap_count:
    raise RuntimeError(f'Day 34 calibration/evaluation overlap: {day34_overlap_count}')

# Resolve canonical NPZ and frozen model from Day 34 input evidence.
npz_name = Path(day34_input_gate['npz_path']).name
model_name = Path(day34_input_gate['model_path']).name
NPZ_PATH = resolve_artifact(
    override_env='DAY35_NPZ_PATH',
    exact_path=Path(day34_input_gate['npz_path']),
    search_root=DATASET_ROOT,
    filename=npz_name,
)
BASELINE_MODEL_PATH = resolve_artifact(
    override_env='DAY35_BASELINE_MODEL_PATH',
    exact_path=Path(day34_input_gate['model_path']),
    search_root=DATASET_ROOT,
    filename=model_name,
)

NPZ_SHA256 = sha256_file(NPZ_PATH)
BASELINE_MODEL_SHA256 = sha256_file(BASELINE_MODEL_PATH)
DAY34_ADAPTER_SHA256 = sha256_file(DAY34_ADAPTER_BUNDLE_PATH)

if adapter_bundle.get('npz_sha256') and adapter_bundle['npz_sha256'] != NPZ_SHA256:
    raise RuntimeError('NPZ checksum không khớp Day 34 adapter bundle.')
if adapter_bundle.get('baseline_model_sha256') and (
    adapter_bundle['baseline_model_sha256'] != BASELINE_MODEL_SHA256
):
    raise RuntimeError('Baseline model checksum không khớp Day 34 adapter bundle.')

validation_subjects = sorted(validation_predictions_df['subject_id'].astype(str).unique().tolist())
if len(validation_subjects) != PROFILE['expected_validation_subjects']:
    raise RuntimeError(
        f'Expected {PROFILE["expected_validation_subjects"]} validation subjects, '
        f'got {len(validation_subjects)}.'
    )

INPUT_GATE_JSON = RUNTIME_ROOT / 'day35-input-gate.json'
input_gate = {
    'schema_version': 'day35-input-gate.v2',
    'created_at_utc': utc_now_iso(),
    'status': 'PASS',
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': day34_protocol.get('dataset_view_id'),
    'class_order': class_order.tolist(),
    'day34_gate_path': str(DAY34_GATE_PATH),
    'day34_gate_sha256': sha256_file(DAY34_GATE_PATH),
    'day34_adapter_bundle_path': str(DAY34_ADAPTER_BUNDLE_PATH),
    'day34_adapter_bundle_sha256': DAY34_ADAPTER_SHA256,
    'day34_validation_predictions_path': str(DAY34_VALIDATION_PREDICTIONS_PATH),
    'day34_validation_predictions_sha256': sha256_file(DAY34_VALIDATION_PREDICTIONS_PATH),
    'npz_path': str(NPZ_PATH),
    'npz_sha256': NPZ_SHA256,
    'baseline_model_path': str(BASELINE_MODEL_PATH),
    'baseline_model_sha256': BASELINE_MODEL_SHA256,
    'validation_subject_ids': validation_subjects,
    'checks': {
        'day34_gate_pass': True,
        'dataset_profile_matches': True,
        'class_contract_matches': True,
        'day34_calibration_evaluation_disjoint': bool(day34_overlap_count == 0),
        'validation_predictions_finite': True,
        'validation_not_used_for_day34_policy_selection': bool(
            day34_policy_selection.get('validation_used_for_policy_selection') is False
        ),
    },
}
write_json_atomic(input_gate, INPUT_GATE_JSON)
persist_artifact(INPUT_GATE_JSON)

print('=' * 96)
print('[CELL 3] DAY 34 INPUT GATE')
print('=' * 96)
print(f'Day 34 gate                    : {DAY34_GATE_PATH}')
print(f'Day 34 adapter bundle          : {DAY34_ADAPTER_BUNDLE_PATH}')
print(f'Canonical NPZ                  : {NPZ_PATH}')
print(f'Frozen baseline model          : {BASELINE_MODEL_PATH}')
print(f'Class order                    : {class_order.tolist()}')
print(f'Validation subjects            : {len(validation_subjects)}')
print(f'Validation episode rows        : {len(validation_predictions_df)}')
print(f'Day 34 calibration/eval overlap: {day34_overlap_count}')
print('[PASS] Day 34 artifacts validated.')

[CELL 3] DAY 34 INPUT GATE
Day 34 gate                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/day34-final-gate.json
Day 34 adapter bundle          : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day34-personalization/day34-personalization-grabmyo-v2/few-shot-adapter-bundle.joblib
Canonical NPZ                  : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/grabmyo-primary4-forearm16-fall14.npz
Frozen baseline model          : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/baseline/grabmyo-primary4-baseline-v1/grabmyo-selected-model.joblib
Class order                    : ['rest', 'hand_close', 'wrist_flexion', 'wrist_extension']
Validation subjects            : 9
Validation episode rows        : 26640
Day 34 calibration/eval overlap: 0
[PASS] Day 34 artifacts validated.


## Bước 4 — Reconstruct frozen P0/P1 representation cho training subjects

**Input:** canonical NPZ, frozen Day 32 model và frozen Day 34 adapter bundle.  
**Output:** trial-level global probabilities, frozen embeddings và prototypes. Không refit baseline, scaler hoặc personalization policy.

In [17]:
# === CELL 4: Reconstruct frozen trial representation ===
feature_indices = np.asarray(adapter_bundle['feature_indices'], dtype=np.int64)
adapter_scaler = adapter_bundle['adapter_scaler']
global_class_prototypes = np.asarray(
    adapter_bundle['global_class_prototypes'],
    dtype=np.float64,
)
selected_policies_raw = adapter_bundle['selected_policies']
selected_policies = {
    int(k): value
    for k, value in selected_policies_raw.items()
}

with np.load(NPZ_PATH, allow_pickle=False) as data:
    common_required = {
        'X', 'y', 'subject_id', 'window_id', 'split_names',
        'feature_names', 'class_order', 'dataset_view_id',
    }
    profile_required = (
        {'repetition_ids', 'record_id'}
        if DATASET_PROFILE == 'mendeley'
        else {'record_id', 'repetition_id', 'session_id', 'trial_id'}
    )
    missing = (common_required | profile_required) - set(data.files)
    if missing:
        raise RuntimeError(f'Canonical NPZ thiếu keys: {sorted(missing)}')

    X = np.asarray(data['X'], dtype=np.float32)
    y = np.asarray(data['y']).astype(str)
    subject_ids = np.asarray(data['subject_id']).astype(str)
    window_ids = np.asarray(data['window_id']).astype(str)
    split_names = np.asarray(data['split_names']).astype(str)
    feature_names = np.asarray(data['feature_names']).astype(str)
    npz_class_order = np.asarray(data['class_order']).astype(str)

    if DATASET_PROFILE == 'mendeley':
        trial_keys = np.asarray(data['repetition_ids']).astype(str)
        record_ids = np.asarray(data['record_id']).astype(str)
        repetition_ids = trial_keys.copy()
        session_ids = np.zeros(len(y), dtype=np.int16)
        trial_ordinals = np.full(len(y), -1, dtype=np.int16)
    else:
        trial_keys = np.asarray(data['record_id']).astype(str)
        record_ids = trial_keys.copy()
        repetition_ids = np.asarray(data['repetition_id']).astype(str)
        session_ids = np.asarray(data['session_id']).astype(np.int16)
        trial_ordinals = np.asarray(data['trial_id']).astype(np.int16)

if not np.array_equal(npz_class_order, class_order):
    raise RuntimeError('NPZ class_order không khớp Day 34 protocol.')
if X.ndim != 2 or not np.isfinite(X).all():
    raise RuntimeError(f'Canonical X không hợp lệ: shape={X.shape}.')
if feature_indices.min() < 0 or feature_indices.max() >= X.shape[1]:
    raise RuntimeError('Day 34 feature indices vượt ngoài canonical X.')
if len({len(X), len(y), len(subject_ids), len(window_ids), len(split_names), len(trial_keys)}) != 1:
    raise RuntimeError('Canonical arrays không cùng số dòng.')
if len(np.unique(window_ids)) != len(window_ids):
    raise RuntimeError('Canonical NPZ chứa duplicate window_id.')

train_mask = split_names == 'train'
validation_mask = split_names == 'validation'
if np.any(~(train_mask | validation_mask)):
    raise RuntimeError(f'Unexpected split names: {np.unique(split_names).tolist()}')

train_subjects = sorted(set(subject_ids[train_mask].tolist()))
npz_validation_subjects = sorted(set(subject_ids[validation_mask].tolist()))
if set(train_subjects) & set(npz_validation_subjects):
    raise RuntimeError('Subject leakage trong canonical split.')
if npz_validation_subjects != validation_subjects:
    raise RuntimeError('Validation subject IDs của NPZ không khớp Day 34 predictions.')
if len(train_subjects) != PROFILE['expected_train_subjects']:
    raise RuntimeError(
        f'Expected {PROFILE["expected_train_subjects"]} train subjects, got {len(train_subjects)}.'
    )

frozen_model = joblib.load(BASELINE_MODEL_PATH)
if set(estimator_classes(frozen_model)) != set(class_order):
    raise RuntimeError('Frozen model classes không khớp class_order.')

X_selected = X[:, feature_indices]
window_predictions = np.asarray(frozen_model.predict(X_selected)).astype(str)
window_scores, window_score_kind = aligned_score_matrix(
    frozen_model,
    X_selected,
    class_order,
)

window_frame = pd.DataFrame({
    'row_index': np.arange(len(y), dtype=np.int64),
    'trial_key': trial_keys,
    'record_id': record_ids,
    'repetition_id': repetition_ids,
    'subject_id': subject_ids,
    'session_id': session_ids,
    'trial_ordinal': trial_ordinals,
    'split_name': split_names,
    'y_true': y,
    'window_id': window_ids,
    'window_pred': window_predictions,
})
score_columns = [f'window_score__{label}' for label in class_order]
for class_index, column in enumerate(score_columns):
    window_frame[column] = window_scores[:, class_index]

trial_rows: list[dict[str, Any]] = []
for trial_key, group in window_frame.groupby('trial_key', sort=False):
    unique_checks = {
        'subject_id': group['subject_id'].unique(),
        'session_id': group['session_id'].unique(),
        'split_name': group['split_name'].unique(),
        'y_true': group['y_true'].unique(),
        'record_id': group['record_id'].unique(),
        'repetition_id': group['repetition_id'].unique(),
    }
    if any(len(values) != 1 for values in unique_checks.values()):
        raise RuntimeError(f'Inconsistent trial metadata: {trial_key}')

    if PROFILE['p0_aggregation'] == 'window_vote_fraction':
        counts = np.asarray(
            [(group['window_pred'] == label).sum() for label in class_order],
            dtype=np.float64,
        )
        raw_trial_scores = counts / counts.sum()
        trial_score_kind = 'window_vote_fraction'
    else:
        raw_trial_scores = group[score_columns].to_numpy(dtype=np.float64).mean(axis=0)
        trial_score_kind = window_score_kind

    p0_pred = str(class_order[int(np.argmax(raw_trial_scores))])
    row = {
        'trial_key': str(trial_key),
        'record_id': str(unique_checks['record_id'][0]),
        'repetition_id': str(unique_checks['repetition_id'][0]),
        'subject_id': str(unique_checks['subject_id'][0]),
        'session_id': int(unique_checks['session_id'][0]),
        'trial_ordinal': int(group['trial_ordinal'].iloc[0]),
        'split_name': str(unique_checks['split_name'][0]),
        'y_true': str(unique_checks['y_true'][0]),
        'valid_window_count': int(len(group)),
        'p0_pred': p0_pred,
        'global_score_kind': trial_score_kind,
    }
    for class_index, label in enumerate(class_order):
        row[f'global_raw__{label}'] = float(raw_trial_scores[class_index])
    trial_rows.append(row)

trial_df = pd.DataFrame(trial_rows).sort_values(
    ['split_name', 'subject_id', 'session_id', 'trial_key']
).reset_index(drop=True)

raw_global_matrix = trial_df[
    [f'global_raw__{label}' for label in class_order]
].to_numpy(dtype=np.float64)
global_prob_matrix = normalize_global_trial_scores(
    raw_global_matrix,
    trial_df['global_score_kind'].iloc[0],
)
for class_index, label in enumerate(class_order):
    trial_df[f'global_prob__{label}'] = global_prob_matrix[:, class_index]

# Rebuild Day 34 frozen trial embeddings using the already-fitted adapter scaler.
trial_position_map = {
    trial_key: index
    for index, trial_key in enumerate(trial_df['trial_key'].astype(str))
}
window_trial_positions = pd.Series(trial_keys).map(trial_position_map).to_numpy()
if np.any(pd.isna(window_trial_positions)):
    raise RuntimeError('Không map được windows sang trial table.')
window_trial_positions = window_trial_positions.astype(np.int64)

embedding_dim = len(feature_indices)
trial_embedding_sum = np.zeros((len(trial_df), embedding_dim), dtype=np.float64)
trial_embedding_count = np.zeros(len(trial_df), dtype=np.int64)
for start in range(0, len(X_selected), CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, len(X_selected))
    transformed = adapter_scaler.transform(X_selected[start:end]).astype(np.float32, copy=False)
    positions = window_trial_positions[start:end]
    np.add.at(trial_embedding_sum, positions, transformed)
    np.add.at(trial_embedding_count, positions, 1)

if np.any(trial_embedding_count <= 0):
    raise RuntimeError('Có trial không có window để tạo embedding.')
trial_embeddings = trial_embedding_sum / trial_embedding_count[:, None]
if not np.isfinite(trial_embeddings).all():
    raise RuntimeError('Frozen trial embeddings chứa NaN/Inf.')
if global_class_prototypes.shape != (len(class_order), embedding_dim):
    raise RuntimeError(
        f'Global prototype shape mismatch: {global_class_prototypes.shape}'
    )

print('=' * 96)
print('[CELL 4] FROZEN REPRESENTATION')
print('=' * 96)
print(f'Canonical windows              : {len(X)}')
print(f'Trial/repetition rows          : {len(trial_df)}')
print(f'Train subjects                 : {len(train_subjects)}')
print(f'Validation subjects            : {len(validation_subjects)}')
print(f'Feature dimension              : {embedding_dim}')
print(f'Trial embedding shape          : {trial_embeddings.shape}')
print(f'Global prototype shape         : {global_class_prototypes.shape}')
print('[PASS] Frozen Day 32/34 representation reconstructed without refit.')

[CELL 4] FROZEN REPRESENTATION
Canonical windows              : 173376
Trial/repetition rows          : 3612
Train subjects                 : 34
Validation subjects            : 9
Feature dimension              : 224
Trial embedding shape          : (3612, 224)
Global prototype shape         : (4, 224)
[PASS] Frozen Day 32/34 representation reconstructed without refit.


## Bước 5 — Tạo ba development partitions và held-out episodes

**Input:** training subjects và frozen Day 34 policies.  
**Output:** predictions trên các held-out trials của training subjects, chia thành ba partition độc lập:

```text
calibration_fit
calibrator_select
threshold_select
```

In [18]:
# === CELL 5: Development subject partitions and frozen episodic predictions ===
ordered_train_subjects = sorted(
    train_subjects,
    key=lambda subject: stable_seed('day35-subject-partition', DATASET_PROFILE, subject),
)
n_train_subjects = len(ordered_train_subjects)
n_calibration_fit = int(math.floor(0.60 * n_train_subjects))
n_calibrator_select = int(round(0.20 * n_train_subjects))
n_threshold_select = n_train_subjects - n_calibration_fit - n_calibrator_select

if min(n_calibration_fit, n_calibrator_select, n_threshold_select) < 5:
    raise RuntimeError(
        'Không đủ training subjects để tách 3 development partitions an toàn.'
    )

subject_partition_map: dict[str, str] = {}
for subject in ordered_train_subjects[:n_calibration_fit]:
    subject_partition_map[subject] = 'calibration_fit'
for subject in ordered_train_subjects[
    n_calibration_fit:n_calibration_fit + n_calibrator_select
]:
    subject_partition_map[subject] = 'calibrator_select'
for subject in ordered_train_subjects[
    n_calibration_fit + n_calibrator_select:
]:
    subject_partition_map[subject] = 'threshold_select'

partition_frame = pd.DataFrame({
    'subject_id': ordered_train_subjects,
    'day35_development_partition': [
        subject_partition_map[subject] for subject in ordered_train_subjects
    ],
})
if partition_frame['subject_id'].nunique() != len(train_subjects):
    raise RuntimeError('Development subject partition không cover đủ training subjects.')
if set(partition_frame['subject_id']) & set(validation_subjects):
    raise RuntimeError('Validation subject lọt vào Day 35 development partitions.')

global_prob_columns = [f'global_prob__{label}' for label in class_order]


def build_episode(subject_id: str, k: int, seed: int) -> dict[str, Any]:
    subject_rows = np.flatnonzero(
        trial_df['subject_id'].astype(str).to_numpy() == str(subject_id)
    )
    if PROFILE['calibration_pool'] == 'session1_only':
        pool_rows = subject_rows[
            trial_df.iloc[subject_rows]['session_id'].to_numpy(dtype=int) == 1
        ]
    else:
        pool_rows = subject_rows

    calibration_rows: list[int] = []
    calibration_by_class: dict[str, list[str]] = {}
    for label in class_order:
        candidates = pool_rows[
            trial_df.iloc[pool_rows]['y_true'].astype(str).to_numpy() == label
        ]
        if len(candidates) < k:
            raise RuntimeError(
                f'Subject={subject_id}, class={label}: '
                f'{len(candidates)} candidates, cần k={k}.'
            )
        rng = np.random.default_rng(
            stable_seed(DATASET_PROFILE, subject_id, label, k, seed)
        )
        selected = np.sort(rng.choice(candidates, size=k, replace=False))
        calibration_rows.extend(selected.tolist())
        calibration_by_class[str(label)] = (
            trial_df.iloc[selected]['trial_key'].astype(str).tolist()
        )

    calibration_rows_array = np.asarray(sorted(calibration_rows), dtype=np.int64)
    evaluation_rows = np.setdiff1d(
        subject_rows,
        calibration_rows_array,
        assume_unique=False,
    )
    calibration_ids = set(
        trial_df.iloc[calibration_rows_array]['trial_key'].astype(str)
    )
    evaluation_ids = set(
        trial_df.iloc[evaluation_rows]['trial_key'].astype(str)
    )
    if calibration_ids & evaluation_ids:
        raise RuntimeError('Calibration/evaluation trial overlap.')

    scopes = {'all_eval': evaluation_rows}
    if DATASET_PROFILE == 'grabmyo':
        eval_sessions = trial_df.iloc[evaluation_rows]['session_id'].to_numpy(dtype=int)
        scopes['same_session_holdout'] = evaluation_rows[eval_sessions == 1]
        scopes['cross_session'] = evaluation_rows[eval_sessions > 1]

    return {
        'subject_id': str(subject_id),
        'k': int(k),
        'seed': int(seed),
        'episode_id': f'day35-{DATASET_PROFILE}-S{subject_id}-K{k}-seed{seed}',
        'calibration_rows': calibration_rows_array,
        'evaluation_rows': evaluation_rows,
        'scopes': scopes,
        'calibration_by_class': calibration_by_class,
    }


def distance_logits(
    evaluation_embeddings: np.ndarray,
    prototypes: np.ndarray,
    metric: str,
    temperature: float,
) -> np.ndarray:
    if metric == 'sqeuclidean':
        distances = np.mean(
            (evaluation_embeddings[:, None, :] - prototypes[None, :, :]) ** 2,
            axis=2,
        )
    elif metric == 'cosine':
        eval_norm = evaluation_embeddings / np.clip(
            np.linalg.norm(evaluation_embeddings, axis=1, keepdims=True),
            1e-12,
            None,
        )
        proto_norm = prototypes / np.clip(
            np.linalg.norm(prototypes, axis=1, keepdims=True),
            1e-12,
            None,
        )
        distances = 1.0 - eval_norm @ proto_norm.T
    else:
        raise ValueError(f'Unsupported distance metric: {metric}')
    return -distances / float(temperature)


def evaluate_episode(
    episode: dict[str, Any],
    policy: dict[str, Any],
) -> pd.DataFrame:
    calibration_rows = episode['calibration_rows']
    evaluation_rows = episode['evaluation_rows']

    subject_prototypes = np.zeros_like(global_class_prototypes)
    for class_index, label in enumerate(class_order):
        class_calibration_rows = calibration_rows[
            trial_df.iloc[calibration_rows]['y_true'].astype(str).to_numpy() == label
        ]
        subject_prototypes[class_index] = (
            trial_embeddings[class_calibration_rows].mean(axis=0)
        )

    rho = float(policy['subject_prototype_weight'])
    personalized_prototypes = (
        rho * subject_prototypes
        + (1.0 - rho) * global_class_prototypes
    )
    logits = distance_logits(
        trial_embeddings[evaluation_rows],
        personalized_prototypes,
        metric=str(policy['distance_metric']),
        temperature=float(policy['temperature']),
    )
    prototype_probabilities = softmax_rows(logits)
    global_probabilities = trial_df.iloc[evaluation_rows][
        global_prob_columns
    ].to_numpy(dtype=np.float64)
    alpha = float(policy['personalization_blend_weight'])
    personalized_probabilities = normalize_probabilities(
        (1.0 - alpha) * global_probabilities
        + alpha * prototype_probabilities
    )

    base = trial_df.iloc[evaluation_rows][[
        'trial_key', 'record_id', 'repetition_id', 'subject_id',
        'session_id', 'trial_ordinal', 'split_name', 'y_true',
        'valid_window_count',
    ]].copy().reset_index(drop=True)
    base['episode_id'] = episode['episode_id']
    base['k'] = episode['k']
    base['seed'] = episode['seed']
    base['eval_scope'] = 'all_eval'
    if DATASET_PROFILE == 'grabmyo':
        base['eval_scope'] = np.where(
            base['session_id'].astype(int) == 1,
            'same_session_holdout',
            'cross_session',
        )
    for class_index, label in enumerate(class_order):
        base[f'p0_prob__{label}'] = global_probabilities[:, class_index]
        base[f'p1_prob__{label}'] = personalized_probabilities[:, class_index]
    base['p0_pred'] = class_order[np.argmax(global_probabilities, axis=1)]
    base['p1_pred'] = class_order[np.argmax(personalized_probabilities, axis=1)]
    return base


development_prediction_parts: list[pd.DataFrame] = []
development_manifest_rows: list[dict[str, Any]] = []
if QUICK_SMOKE_TEST:
    development_subjects = []
    for partition_name in (
        'calibration_fit',
        'calibrator_select',
        'threshold_select',
    ):
        development_subjects.extend(
            partition_frame.loc[
                partition_frame['day35_development_partition'] == partition_name,
                'subject_id',
            ].astype(str).tolist()[:2]
        )
else:
    development_subjects = ordered_train_subjects

for k in K_VALUES:
    policy = selected_policies[int(k)]
    for subject_id in development_subjects:
        episode = build_episode(subject_id, k, DAY35_EPISODE_SEED)
        for label, trial_list in episode['calibration_by_class'].items():
            for trial_key in trial_list:
                development_manifest_rows.append({
                    'episode_id': episode['episode_id'],
                    'subject_id': str(subject_id),
                    'day35_development_partition': subject_partition_map[subject_id],
                    'k': int(k),
                    'class_label': str(label),
                    'calibration_trial_key': str(trial_key),
                })
        predictions = evaluate_episode(episode, policy)
        predictions['day35_development_partition'] = subject_partition_map[subject_id]
        development_prediction_parts.append(predictions)

development_predictions_df = pd.concat(development_prediction_parts, ignore_index=True)
development_manifest_df = pd.DataFrame(development_manifest_rows)

if set(development_predictions_df['subject_id']) & set(validation_subjects):
    raise RuntimeError('Validation subject xuất hiện trong development predictions.')
if not set(development_predictions_df['split_name'].astype(str)) == {'train'}:
    raise RuntimeError('Development predictions chứa non-train rows.')

for arm in ARMS:
    probability_columns = [f'{arm}_prob__{label}' for label in class_order]
    matrix = development_predictions_df[probability_columns].to_numpy(dtype=np.float64)
    if not np.isfinite(matrix).all():
        raise RuntimeError(f'Development {arm} probabilities chứa NaN/Inf.')
    if not np.allclose(matrix.sum(axis=1), 1.0, atol=1e-8):
        raise RuntimeError(f'Development {arm} probabilities không sum về 1.')

PARTITION_CSV = RUNTIME_ROOT / 'day35-development-subject-partitions.csv'
DEVELOPMENT_MANIFEST_CSV = RUNTIME_ROOT / 'day35-development-episode-manifest.csv'
DEVELOPMENT_PREDICTIONS_CSV_GZ = RUNTIME_ROOT / 'day35-development-predictions.csv.gz'
write_dataframe_atomic(partition_frame, PARTITION_CSV)
write_dataframe_atomic(development_manifest_df, DEVELOPMENT_MANIFEST_CSV)
write_dataframe_atomic(development_predictions_df, DEVELOPMENT_PREDICTIONS_CSV_GZ)
for path in (
    PARTITION_CSV,
    DEVELOPMENT_MANIFEST_CSV,
    DEVELOPMENT_PREDICTIONS_CSV_GZ,
):
    persist_artifact(path)

print('=' * 96)
print('[CELL 5] DAY 35 DEVELOPMENT EPISODES')
print('=' * 96)
print(partition_frame['day35_development_partition'].value_counts().to_string())
print(f'Development subjects           : {len(development_subjects)}')
print(f'Development prediction rows    : {len(development_predictions_df)}')
print(f'Development calibration rows   : {len(development_manifest_df)}')
print('[PASS] Three disjoint training-subject development partitions created.')

[CELL 5] DAY 35 DEVELOPMENT EPISODES
day35_development_partition
calibration_fit      20
calibrator_select     7
threshold_select      7
Development subjects           : 34
Development prediction rows    : 5032
Development calibration rows   : 680
[PASS] Three disjoint training-subject development partitions created.


## Bước 6 — Fit và chọn calibrator

**Input:** `calibration_fit` và `calibrator_select`.  
**Output:** calibrator được chọn riêng cho mỗi `(k, arm)`.

Candidate chính:

- `identity`;
- `scalar_temperature`.

Temperature scaling được fit bằng subject-equal weighting và giữ nguyên argmax.

In [19]:
# === CELL 6: Fit and select calibration method without validation ===
calibrator_registry: dict[str, Any] = {}
candidate_metric_rows: list[dict[str, Any]] = []
calibrator_selection_payload: dict[str, Any] = {
    'schema_version': 'day35-calibrator-selection.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'fit_partition': 'calibration_fit',
    'selection_partition': 'calibrator_select',
    'validation_subjects_used': [],
    'selections': {},
}

for k in K_VALUES:
    for arm in ARMS:
        fit_frame = development_predictions_df[
            (development_predictions_df['k'].astype(int) == int(k))
            & (
                development_predictions_df['day35_development_partition']
                == 'calibration_fit'
            )
        ].copy()
        select_frame = development_predictions_df[
            (development_predictions_df['k'].astype(int) == int(k))
            & (
                development_predictions_df['day35_development_partition']
                == 'calibrator_select'
            )
        ].copy()
        if fit_frame.empty or select_frame.empty:
            raise RuntimeError(f'Empty calibration partitions for k={k}, arm={arm}.')

        probability_columns = [f'{arm}_prob__{label}' for label in class_order]
        fit_probabilities = fit_frame[probability_columns].to_numpy(dtype=np.float64)
        select_probabilities = select_frame[probability_columns].to_numpy(dtype=np.float64)
        fit_y = labels_to_indices(fit_frame['y_true'], class_order)
        select_y = labels_to_indices(select_frame['y_true'], class_order)
        fit_weights = subject_equal_weights(fit_frame['subject_id'])
        select_weights = subject_equal_weights(select_frame['subject_id'])

        identity = IdentityCalibrator().fit(
            fit_probabilities,
            fit_y,
            fit_weights,
        )
        temperature = ScalarTemperatureCalibrator().fit(
            fit_probabilities,
            fit_y,
            fit_weights,
        )
        candidates = {
            'identity': identity,
            'scalar_temperature': temperature,
        }

        arm_rows = []
        for method_name, calibrator in candidates.items():
            select_calibrated = calibrator.predict_proba(select_probabilities)
            fit_calibrated = calibrator.predict_proba(fit_probabilities)
            fit_metrics = calibration_metrics(fit_y, fit_calibrated, fit_weights)
            select_metrics = calibration_metrics(
                select_y,
                select_calibrated,
                select_weights,
            )
            argmax_preserved = bool(np.array_equal(
                np.argmax(select_probabilities, axis=1),
                np.argmax(select_calibrated, axis=1),
            ))
            row = {
                'k': int(k),
                'arm': arm,
                'method': method_name,
                'fit_subject_count': int(fit_frame['subject_id'].nunique()),
                'selection_subject_count': int(select_frame['subject_id'].nunique()),
                'fit_row_count': int(len(fit_frame)),
                'selection_row_count': int(len(select_frame)),
                'argmax_preserved': argmax_preserved,
                **{f'fit_{name}': value for name, value in fit_metrics.items()},
                **{f'selection_{name}': value for name, value in select_metrics.items()},
                **calibrator.metadata(),
            }
            arm_rows.append(row)
            candidate_metric_rows.append(row)

        arm_candidate_frame = pd.DataFrame(arm_rows)
        identity_row = arm_candidate_frame[
            arm_candidate_frame['method'] == 'identity'
        ].iloc[0]
        temperature_row = arm_candidate_frame[
            arm_candidate_frame['method'] == 'scalar_temperature'
        ].iloc[0]
        nll_gain = float(
            identity_row['selection_nll']
            - temperature_row['selection_nll']
        )
        if nll_gain >= MIN_TEMPERATURE_NLL_GAIN:
            selected_method = 'scalar_temperature'
        else:
            selected_method = 'identity'

        selected_calibrator = candidates[selected_method]
        key = f'k{k}:{arm}'
        calibrator_registry[key] = selected_calibrator
        selected_row = arm_candidate_frame[
            arm_candidate_frame['method'] == selected_method
        ].iloc[0].to_dict()
        calibrator_selection_payload['selections'][key] = {
            'k': int(k),
            'arm': arm,
            'selected_method': selected_method,
            'temperature_nll_gain_vs_identity': nll_gain,
            'minimum_required_nll_gain': MIN_TEMPERATURE_NLL_GAIN,
            'selected_metadata': selected_calibrator.metadata(),
            'selection_metrics': {
                name.replace('selection_', ''): value
                for name, value in selected_row.items()
                if name.startswith('selection_')
            },
            'argmax_preserved': bool(selected_row['argmax_preserved']),
        }

calibrator_candidate_metrics_df = pd.DataFrame(candidate_metric_rows)
if not calibrator_candidate_metrics_df['argmax_preserved'].all():
    raise RuntimeError('Có calibrator candidate thay đổi argmax.')

CALIBRATOR_CANDIDATES_CSV = RUNTIME_ROOT / 'day35-calibrator-candidate-metrics.csv'
CALIBRATOR_SELECTION_JSON = RUNTIME_ROOT / 'day35-calibrator-selection.json'
write_dataframe_atomic(calibrator_candidate_metrics_df, CALIBRATOR_CANDIDATES_CSV)
write_json_atomic(calibrator_selection_payload, CALIBRATOR_SELECTION_JSON)
for path in (CALIBRATOR_CANDIDATES_CSV, CALIBRATOR_SELECTION_JSON):
    persist_artifact(path)

print('=' * 96)
print('[CELL 6] CALIBRATOR SELECTION')
print('=' * 96)
for key, selection in calibrator_selection_payload['selections'].items():
    metadata = selection['selected_metadata']
    print(
        f"{key:<10} method={selection['selected_method']:<20} "
        f"metadata={metadata}"
    )
print('[PASS] Calibrators fit and selected without validation data.')

[CELL 6] CALIBRATOR SELECTION
k2:p0      method=scalar_temperature   metadata={'method': 'scalar_temperature', 'temperature': 0.22375466297877883}
k2:p1      method=scalar_temperature   metadata={'method': 'scalar_temperature', 'temperature': 0.17008595372327376}
k3:p0      method=scalar_temperature   metadata={'method': 'scalar_temperature', 'temperature': 0.21357499959267473}
k3:p1      method=scalar_temperature   metadata={'method': 'scalar_temperature', 'temperature': 0.16158212204017794}
[PASS] Calibrators fit and selected without validation data.


## Bước 7 — Chọn confidence score và freeze abstention operating points

**Input:** chỉ `threshold_select`.  
**Output:** score được chọn bằng subject-mean AURC và ba operating points:

- `coverage_95`;
- `coverage_90`;
- `coverage_80`.

Threshold là empirical quantile trên `threshold_select`; validation không tham gia.

In [20]:
# === CELL 7: Select confidence score and freeze abstention thresholds ===
score_selection_rows: list[dict[str, Any]] = []
development_curve_parts: list[pd.DataFrame] = []
abstention_policy_payload: dict[str, Any] = {
    'schema_version': 'day35-abstention-policy.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'selection_partition': 'threshold_select',
    'validation_subjects_used': [],
    'primary_operating_point': PRIMARY_OPERATING_POINT,
    'coverage_targets': list(OPERATING_COVERAGE_TARGETS),
    'policies': {},
}

for k in K_VALUES:
    for arm in ARMS:
        threshold_frame = development_predictions_df[
            (development_predictions_df['k'].astype(int) == int(k))
            & (
                development_predictions_df['day35_development_partition']
                == 'threshold_select'
            )
        ].copy()
        probability_columns = [f'{arm}_prob__{label}' for label in class_order]
        uncalibrated = threshold_frame[probability_columns].to_numpy(dtype=np.float64)
        calibrated = calibrator_registry[f'k{k}:{arm}'].predict_proba(uncalibrated)
        y_index = labels_to_indices(threshold_frame['y_true'], class_order)
        score_map = confidence_scores(calibrated)

        candidate_rows = []
        for score_name, score in score_map.items():
            subject_aurcs = []
            for subject_id, subject_group in threshold_frame.groupby('subject_id'):
                indices = subject_group.index.to_numpy()
                local_positions = threshold_frame.index.get_indexer(indices)
                subject_aurcs.append(
                    exact_aurc(
                        y_index[local_positions],
                        calibrated[local_positions],
                        score[local_positions],
                    )
                )
            pooled_aurc = exact_aurc(y_index, calibrated, score)
            subject_mean_aurc = float(np.mean(subject_aurcs))
            row = {
                'k': int(k),
                'arm': arm,
                'score_name': score_name,
                'threshold_subject_count': int(
                    threshold_frame['subject_id'].nunique()
                ),
                'threshold_row_count': int(len(threshold_frame)),
                'pooled_aurc': pooled_aurc,
                'subject_mean_aurc': subject_mean_aurc,
            }
            candidate_rows.append(row)
            score_selection_rows.append(row)

            curve = risk_coverage_curve(y_index, calibrated, score)
            curve.insert(0, 'score_name', score_name)
            curve.insert(0, 'arm', arm)
            curve.insert(0, 'k', int(k))
            development_curve_parts.append(curve)

        candidate_frame = pd.DataFrame(candidate_rows).sort_values(
            ['subject_mean_aurc', 'pooled_aurc', 'score_name'],
            ascending=[True, True, True],
        ).reset_index(drop=True)
        selected_score_name = str(candidate_frame.iloc[0]['score_name'])
        selected_score = score_map[selected_score_name]

        operating_points: dict[str, Any] = {
            'no_abstention': {
                'target_coverage': 1.0,
                'threshold': 0.0,
            }
        }
        for target_coverage in OPERATING_COVERAGE_TARGETS:
            point_name = f'coverage_{int(round(target_coverage * 100))}'
            threshold = quantile_threshold_for_coverage(
                selected_score,
                target_coverage,
            )
            accepted = selected_score >= threshold
            temp_frame = threshold_frame.copy()
            temp_frame['accepted'] = accepted
            metrics = selective_metrics_from_frame(
                temp_frame.assign(
                    calibrated_pred=class_order[np.argmax(calibrated, axis=1)]
                ),
                accepted_column='accepted',
                truth_column='y_true',
                prediction_column='calibrated_pred',
                labels=class_order,
            )
            operating_points[point_name] = {
                'target_coverage': float(target_coverage),
                'threshold': float(threshold),
                'development_observed_metrics': metrics,
            }

        policy_key = f'k{k}:{arm}'
        abstention_policy_payload['policies'][policy_key] = {
            'k': int(k),
            'arm': arm,
            'calibrator_method': calibrator_registry[policy_key].metadata(),
            'selected_score_name': selected_score_name,
            'score_selection_metric': 'subject_mean_aurc',
            'score_candidate_metrics': candidate_frame.to_dict(orient='records'),
            'operating_points': operating_points,
            'threshold_partition': 'threshold_select',
        }

score_selection_df = pd.DataFrame(score_selection_rows)
development_curve_df = pd.concat(development_curve_parts, ignore_index=True)

SCORE_SELECTION_CSV = RUNTIME_ROOT / 'day35-abstention-score-selection.csv'
DEVELOPMENT_CURVE_CSV = RUNTIME_ROOT / 'day35-development-coverage-risk.csv'
ABSTENTION_POLICY_JSON = RUNTIME_ROOT / 'day35-abstention-policy.json'
write_dataframe_atomic(score_selection_df, SCORE_SELECTION_CSV)
write_dataframe_atomic(development_curve_df, DEVELOPMENT_CURVE_CSV)
write_json_atomic(abstention_policy_payload, ABSTENTION_POLICY_JSON)
for path in (SCORE_SELECTION_CSV, DEVELOPMENT_CURVE_CSV, ABSTENTION_POLICY_JSON):
    persist_artifact(path)

print('=' * 96)
print('[CELL 7] FROZEN ABSTENTION POLICY')
print('=' * 96)
for key, policy in abstention_policy_payload['policies'].items():
    primary = policy['operating_points'][PRIMARY_OPERATING_POINT]
    print(
        f"{key:<10} score={policy['selected_score_name']:<30} "
        f"{PRIMARY_OPERATING_POINT} threshold={primary['threshold']:.6f}"
    )
print('[PASS] Confidence scores and thresholds frozen on threshold_select only.')

[CELL 7] FROZEN ABSTENTION POLICY
k2:p0      score=one_minus_normalized_entropy   coverage_90 threshold=0.983806
k2:p1      score=max_probability                coverage_90 threshold=0.997617
k3:p0      score=one_minus_normalized_entropy   coverage_90 threshold=0.988006
k3:p1      score=max_probability                coverage_90 threshold=0.998484
[PASS] Confidence scores and thresholds frozen on threshold_select only.


## Bước 8 — One-time frozen validation evaluation

**Input:** Day 34 validation prediction rows và frozen Day 35 bundle.  
**Output:** calibrated probabilities, abstention decisions, calibration metrics, coverage–risk metrics và subject-level confidence intervals.

In [21]:
# === CELL 8: Apply frozen calibration and abstention policy to validation ===
validation_long_parts: list[pd.DataFrame] = []
validation_calibration_rows: list[dict[str, Any]] = []
validation_selective_rows: list[dict[str, Any]] = []
validation_subject_rows: list[dict[str, Any]] = []
validation_class_rows: list[dict[str, Any]] = []
validation_session_rows: list[dict[str, Any]] = []
validation_reliability_parts: list[pd.DataFrame] = []
validation_curve_parts: list[pd.DataFrame] = []
argmax_change_count = 0

scope_names = ['all_eval']
if DATASET_PROFILE == 'grabmyo':
    scope_names.extend(['same_session_holdout', 'cross_session'])

for k in K_VALUES:
    source_k = validation_predictions_df[
        validation_predictions_df['k'].astype(int) == int(k)
    ].copy()
    for arm in ARMS:
        policy_key = f'k{k}:{arm}'
        probability_columns = [f'{arm}_prob__{label}' for label in class_order]
        uncalibrated = source_k[probability_columns].to_numpy(dtype=np.float64)
        calibrated = calibrator_registry[policy_key].predict_proba(uncalibrated)
        uncalibrated_pred_index = np.argmax(uncalibrated, axis=1)
        calibrated_pred_index = np.argmax(calibrated, axis=1)
        argmax_change_count += int(np.sum(
            uncalibrated_pred_index != calibrated_pred_index
        ))

        policy = abstention_policy_payload['policies'][policy_key]
        selected_score_name = policy['selected_score_name']
        selected_score = confidence_scores(calibrated)[selected_score_name]

        arm_frame = source_k[[
            'episode_id', 'subject_id', 'trial_key', 'record_id',
            'repetition_id', 'session_id', 'split_name', 'y_true',
            'k', 'seed', 'eval_scope', 'valid_window_count',
        ]].copy()
        arm_frame['arm'] = arm
        arm_frame['calibrator_method'] = (
            calibrator_registry[policy_key].metadata()['method']
        )
        arm_frame['uncalibrated_pred'] = class_order[uncalibrated_pred_index]
        arm_frame['calibrated_pred'] = class_order[calibrated_pred_index]
        arm_frame['uncalibrated_confidence'] = np.max(uncalibrated, axis=1)
        arm_frame['calibrated_confidence'] = np.max(calibrated, axis=1)
        arm_frame['selected_score_name'] = selected_score_name
        arm_frame['abstention_score'] = selected_score
        for class_index, label in enumerate(class_order):
            arm_frame[f'uncal_prob__{label}'] = uncalibrated[:, class_index]
            arm_frame[f'cal_prob__{label}'] = calibrated[:, class_index]

        for operating_point, point in policy['operating_points'].items():
            threshold = float(point['threshold'])
            arm_frame[f'accepted__{operating_point}'] = (
                selected_score >= threshold
            )

        validation_long_parts.append(arm_frame)

        for scope_name in scope_names:
            if scope_name == 'all_eval':
                scope_mask = np.ones(len(source_k), dtype=bool)
            else:
                scope_mask = (
                    source_k['eval_scope'].astype(str).to_numpy() == scope_name
                )
            if not np.any(scope_mask):
                continue

            scope_y = labels_to_indices(
                source_k.loc[scope_mask, 'y_true'],
                class_order,
            )
            scope_subjects = source_k.loc[scope_mask, 'subject_id'].astype(str)
            scope_weights = subject_equal_weights(scope_subjects)
            before_metrics = calibration_metrics(
                scope_y,
                uncalibrated[scope_mask],
                scope_weights,
            )
            after_metrics = calibration_metrics(
                scope_y,
                calibrated[scope_mask],
                scope_weights,
            )
            validation_calibration_rows.append({
                'k': int(k),
                'arm': arm,
                'scope': scope_name,
                'subject_count': int(scope_subjects.nunique()),
                'episode_count': int(
                    source_k.loc[scope_mask, 'episode_id'].nunique()
                ),
                'row_count_across_episodes': int(scope_mask.sum()),
                **{f'uncalibrated_{name}': value for name, value in before_metrics.items()},
                **{f'calibrated_{name}': value for name, value in after_metrics.items()},
                **{
                    f'delta_{name}': after_metrics[name] - before_metrics[name]
                    for name in before_metrics
                },
            })

            for calibration_state, probabilities in (
                ('uncalibrated', uncalibrated[scope_mask]),
                ('calibrated', calibrated[scope_mask]),
            ):
                bins = calibration_bin_table(
                    y_index=scope_y,
                    probabilities=probabilities,
                    n_bins=N_CALIBRATION_BINS,
                    strategy='equal_width',
                    weights=scope_weights,
                )
                bins.insert(0, 'calibration_state', calibration_state)
                bins.insert(0, 'scope', scope_name)
                bins.insert(0, 'arm', arm)
                bins.insert(0, 'k', int(k))
                validation_reliability_parts.append(bins)

            curve = risk_coverage_curve(
                scope_y,
                calibrated[scope_mask],
                selected_score[scope_mask],
            )
            curve.insert(0, 'score_name', selected_score_name)
            curve.insert(0, 'scope', scope_name)
            curve.insert(0, 'arm', arm)
            curve.insert(0, 'k', int(k))
            validation_curve_parts.append(curve)

        for operating_point, point in policy['operating_points'].items():
            accepted_column = f'accepted__{operating_point}'
            for scope_name in scope_names:
                if scope_name == 'all_eval':
                    scope_frame = arm_frame.copy()
                else:
                    scope_frame = arm_frame[
                        arm_frame['eval_scope'].astype(str) == scope_name
                    ].copy()
                if scope_frame.empty:
                    continue

                micro_metrics = selective_metrics_from_frame(
                    scope_frame,
                    accepted_column=accepted_column,
                    truth_column='y_true',
                    prediction_column='calibrated_pred',
                    labels=class_order,
                )

                subject_metric_parts = []
                for subject_id, subject_group in scope_frame.groupby('subject_id'):
                    subject_metrics = selective_metrics_from_frame(
                        subject_group,
                        accepted_column=accepted_column,
                        truth_column='y_true',
                        prediction_column='calibrated_pred',
                        labels=class_order,
                    )
                    subject_row = {
                        'k': int(k),
                        'arm': arm,
                        'scope': scope_name,
                        'operating_point': operating_point,
                        'subject_id': str(subject_id),
                        **subject_metrics,
                    }
                    subject_metric_parts.append(subject_row)
                    validation_subject_rows.append(subject_row)

                subject_metrics_frame = pd.DataFrame(subject_metric_parts)
                coverage_ci_low, coverage_ci_high = bootstrap_subject_mean_ci(
                    subject_metrics_frame,
                    'coverage',
                    stable_seed(
                        'day35-bootstrap-coverage',
                        DATASET_PROFILE,
                        k,
                        arm,
                        scope_name,
                        operating_point,
                    ),
                    BOOTSTRAP_RESAMPLES,
                )
                risk_ci_low, risk_ci_high = bootstrap_subject_mean_ci(
                    subject_metrics_frame,
                    'selective_risk',
                    stable_seed(
                        'day35-bootstrap-risk',
                        DATASET_PROFILE,
                        k,
                        arm,
                        scope_name,
                        operating_point,
                    ),
                    BOOTSTRAP_RESAMPLES,
                )
                validation_selective_rows.append({
                    'k': int(k),
                    'arm': arm,
                    'scope': scope_name,
                    'operating_point': operating_point,
                    'threshold': float(point['threshold']),
                    'score_name': selected_score_name,
                    'subject_count': int(scope_frame['subject_id'].nunique()),
                    'episode_count': int(scope_frame['episode_id'].nunique()),
                    **micro_metrics,
                    'subject_mean_coverage': float(
                        subject_metrics_frame['coverage'].mean()
                    ),
                    'subject_mean_coverage_ci95_low': coverage_ci_low,
                    'subject_mean_coverage_ci95_high': coverage_ci_high,
                    'subject_mean_selective_risk': float(
                        subject_metrics_frame['selective_risk'].mean()
                    ),
                    'subject_mean_selective_risk_ci95_low': risk_ci_low,
                    'subject_mean_selective_risk_ci95_high': risk_ci_high,
                })

                for class_label, class_group in scope_frame.groupby('y_true'):
                    class_metrics = selective_metrics_from_frame(
                        class_group,
                        accepted_column=accepted_column,
                        truth_column='y_true',
                        prediction_column='calibrated_pred',
                        labels=class_order,
                    )
                    validation_class_rows.append({
                        'k': int(k),
                        'arm': arm,
                        'scope': scope_name,
                        'operating_point': operating_point,
                        'class_label': str(class_label),
                        **class_metrics,
                    })

                for session_id, session_group in scope_frame.groupby('session_id'):
                    session_metrics = selective_metrics_from_frame(
                        session_group,
                        accepted_column=accepted_column,
                        truth_column='y_true',
                        prediction_column='calibrated_pred',
                        labels=class_order,
                    )
                    validation_session_rows.append({
                        'k': int(k),
                        'arm': arm,
                        'scope': scope_name,
                        'operating_point': operating_point,
                        'session_id': int(session_id),
                        **session_metrics,
                    })

validation_long_df = pd.concat(validation_long_parts, ignore_index=True)
validation_calibration_metrics_df = pd.DataFrame(validation_calibration_rows)
validation_selective_metrics_df = pd.DataFrame(validation_selective_rows)
validation_subject_metrics_df = pd.DataFrame(validation_subject_rows)
validation_class_metrics_df = pd.DataFrame(validation_class_rows)
validation_session_metrics_df = pd.DataFrame(validation_session_rows)
validation_reliability_df = pd.concat(validation_reliability_parts, ignore_index=True)
validation_curve_df = pd.concat(validation_curve_parts, ignore_index=True)

if argmax_change_count != 0:
    raise RuntimeError(
        f'Calibration changed predicted class on {argmax_change_count} rows.'
    )

VALIDATION_PREDICTIONS_CSV_GZ = RUNTIME_ROOT / 'day35-validation-predictions.csv.gz'
VALIDATION_CALIBRATION_CSV = RUNTIME_ROOT / 'day35-validation-calibration-metrics.csv'
VALIDATION_SELECTIVE_CSV = RUNTIME_ROOT / 'day35-validation-selective-metrics.csv'
VALIDATION_SUBJECT_CSV = RUNTIME_ROOT / 'day35-validation-subject-metrics.csv'
VALIDATION_CLASS_CSV = RUNTIME_ROOT / 'day35-validation-class-metrics.csv'
VALIDATION_SESSION_CSV = RUNTIME_ROOT / 'day35-validation-session-metrics.csv'
VALIDATION_RELIABILITY_CSV = RUNTIME_ROOT / 'day35-validation-reliability-bins.csv'
VALIDATION_CURVE_CSV = RUNTIME_ROOT / 'day35-validation-coverage-risk.csv'

write_dataframe_atomic(validation_long_df, VALIDATION_PREDICTIONS_CSV_GZ)
write_dataframe_atomic(validation_calibration_metrics_df, VALIDATION_CALIBRATION_CSV)
write_dataframe_atomic(validation_selective_metrics_df, VALIDATION_SELECTIVE_CSV)
write_dataframe_atomic(validation_subject_metrics_df, VALIDATION_SUBJECT_CSV)
write_dataframe_atomic(validation_class_metrics_df, VALIDATION_CLASS_CSV)
write_dataframe_atomic(validation_session_metrics_df, VALIDATION_SESSION_CSV)
write_dataframe_atomic(validation_reliability_df, VALIDATION_RELIABILITY_CSV)
write_dataframe_atomic(validation_curve_df, VALIDATION_CURVE_CSV)

for path in (
    VALIDATION_PREDICTIONS_CSV_GZ,
    VALIDATION_CALIBRATION_CSV,
    VALIDATION_SELECTIVE_CSV,
    VALIDATION_SUBJECT_CSV,
    VALIDATION_CLASS_CSV,
    VALIDATION_SESSION_CSV,
    VALIDATION_RELIABILITY_CSV,
    VALIDATION_CURVE_CSV,
):
    persist_artifact(path)

print('=' * 96)
print('[CELL 8] FROZEN VALIDATION')
print('=' * 96)
print(f'Validation long rows           : {len(validation_long_df)}')
print(f'Calibration metric rows        : {len(validation_calibration_metrics_df)}')
print(f'Selective metric rows          : {len(validation_selective_metrics_df)}')
print(f'Argmax changes after calibration: {argmax_change_count}')
display(
    validation_selective_metrics_df[
        (validation_selective_metrics_df['scope'] == PROFILE['primary_scope'])
        & (
            validation_selective_metrics_df['operating_point']
            == PRIMARY_OPERATING_POINT
        )
    ][[
        'k', 'arm', 'coverage', 'selective_risk',
        'selective_macro_f1', 'error_capture_rate',
    ]]
)
print('[PASS] Frozen validation evaluated once with frozen Day 35 policies.')

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains cl

[CELL 8] FROZEN VALIDATION
Validation long rows           : 53280
Calibration metric rows        : 12
Selective metric rows          : 48
Argmax changes after calibration: 0


,k,arm,coverage,selective_risk,selective_macro_f1,error_capture_rate
8,2,p0,0.869048,0.0,1.0,1.0
20,2,p1,0.864087,0.0,1.0,1.0
32,3,p0,0.865079,0.0,1.0,1.0
44,3,p1,0.857044,0.0,1.0,1.0


[PASS] Frozen validation evaluated once with frozen Day 35 policies.


## Bước 9 — Reliability và coverage–risk figures

**Input:** frozen validation outputs.  
**Output:** một reliability diagram và một coverage–risk figure cho từng `(k, arm)`.

In [22]:
# === CELL 9: Reliability and coverage-risk figures ===
figure_paths: list[Path] = []

for k in K_VALUES:
    for arm in ARMS:
        primary_scope = PROFILE['primary_scope']

        reliability_subset = validation_reliability_df[
            (validation_reliability_df['k'].astype(int) == int(k))
            & (validation_reliability_df['arm'] == arm)
            & (validation_reliability_df['scope'] == primary_scope)
        ]
        reliability_path = (
            RUNTIME_ROOT / f'day35-reliability-k{k}-{arm}-{primary_scope}.png'
        )
        fig, ax = plt.subplots(figsize=(7, 6))
        for state, state_frame in reliability_subset.groupby('calibration_state'):
            ax.plot(
                state_frame['confidence_mean'],
                state_frame['accuracy_mean'],
                marker='o',
                label=state,
            )
        ax.plot([0, 1], [0, 1], linestyle='--', label='ideal')
        ax.set_xlim(0, 1)
        ax.set_ylim(0, 1)
        ax.set_xlabel('Mean confidence')
        ax.set_ylabel('Empirical accuracy')
        ax.set_title(
            f'Day 35 reliability — {DATASET_PROFILE}, k={k}, {arm}, {primary_scope}'
        )
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(reliability_path, dpi=170, bbox_inches='tight')
        plt.close(fig)
        figure_paths.append(reliability_path)

        curve_subset = validation_curve_df[
            (validation_curve_df['k'].astype(int) == int(k))
            & (validation_curve_df['arm'] == arm)
            & (validation_curve_df['scope'] == primary_scope)
        ].sort_values('coverage')
        curve_path = (
            RUNTIME_ROOT / f'day35-coverage-risk-k{k}-{arm}-{primary_scope}.png'
        )
        fig, ax = plt.subplots(figsize=(7, 6))
        ax.plot(
            curve_subset['coverage'],
            curve_subset['selective_risk'],
            marker='.',
        )
        policy = abstention_policy_payload['policies'][f'k{k}:{arm}']
        for point_name, point in policy['operating_points'].items():
            if point_name == 'no_abstention':
                continue
            row = validation_selective_metrics_df[
                (validation_selective_metrics_df['k'].astype(int) == int(k))
                & (validation_selective_metrics_df['arm'] == arm)
                & (validation_selective_metrics_df['scope'] == primary_scope)
                & (
                    validation_selective_metrics_df['operating_point']
                    == point_name
                )
            ]
            if len(row) == 1:
                ax.scatter(
                    row['coverage'].iloc[0],
                    row['selective_risk'].iloc[0],
                    label=point_name,
                )
        ax.set_xlim(0, 1)
        ax.set_ylim(bottom=0)
        ax.set_xlabel('Coverage')
        ax.set_ylabel('Selective risk')
        ax.set_title(
            f'Day 35 coverage–risk — {DATASET_PROFILE}, k={k}, {arm}, {primary_scope}'
        )
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(curve_path, dpi=170, bbox_inches='tight')
        plt.close(fig)
        figure_paths.append(curve_path)

for path in figure_paths:
    persist_artifact(path)

FIGURE_MANIFEST_JSON = RUNTIME_ROOT / 'day35-figure-manifest.json'
write_json_atomic({
    'schema_version': 'day35-figure-manifest.v1',
    'created_at_utc': utc_now_iso(),
    'figures': [
        {
            'filename': path.name,
            'sha256': sha256_file(path),
            'size_bytes': int(path.stat().st_size),
        }
        for path in figure_paths
    ],
}, FIGURE_MANIFEST_JSON)
persist_artifact(FIGURE_MANIFEST_JSON)

print(f'[PASS] Generated {len(figure_paths)} Day 35 figures.')

[PASS] Generated 8 Day 35 figures.


## Bước 10 — Protocol, deployable bundle, evidence, final gate và handoff ZIP

**Input:** toàn bộ artifact Day 35.  
**Output:** bundle cho Day 36, evidence JSON, final gate và ZIP bàn giao.

In [23]:
# === CELL 10: Bundle, evidence, final gate and handoff ===
PROTOCOL_JSON = RUNTIME_ROOT / 'day35-confidence-abstention-protocol.json'
BUNDLE_JOBLIB = RUNTIME_ROOT / 'day35-calibration-abstention-bundle.joblib'
EVIDENCE_JSON = RUNTIME_ROOT / 'day35-confidence-calibration-evidence.json'
FINAL_GATE_JSON = RUNTIME_ROOT / 'day35-final-gate.json'
HANDOFF_ZIP = RUNTIME_ROOT / 'day35-confidence-calibration-handoff.zip'

protocol_payload = {
    'schema_version': 'day35-confidence-abstention-protocol.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': day34_protocol.get('dataset_view_id'),
    'class_order': class_order.tolist(),
    'k_values': list(K_VALUES),
    'arms': list(ARMS),
    'development_subject_partitions': {
        'calibration_fit': (
            'fit calibration parameters using subject-equal weighting'
        ),
        'calibrator_select': (
            'select identity versus scalar temperature using NLL'
        ),
        'threshold_select': (
            'select confidence score by subject-mean AURC and freeze thresholds'
        ),
    },
    'calibration_methods': ['identity', 'scalar_temperature'],
    'calibration_preserves_argmax': True,
    'confidence_score_candidates': [
        'max_probability',
        'probability_margin',
        'one_minus_normalized_entropy',
    ],
    'operating_coverage_targets': list(OPERATING_COVERAGE_TARGETS),
    'primary_operating_point': PRIMARY_OPERATING_POINT,
    'risk_definition': 'error rate among accepted predictions',
    'coverage_definition': 'accepted predictions divided by all eligible predictions',
    'abstention_is_not_a_class': True,
    'validation_used_for_fit_or_selection': False,
    'test_set_opened': False,
    'clinical_use_allowed': False,
}
write_json_atomic(protocol_payload, PROTOCOL_JSON)

bundle_payload = {
    'schema_version': 'day35-calibration-abstention-bundle.v2',
    'created_at_utc': utc_now_iso(),
    'dataset_profile': DATASET_PROFILE,
    'dataset_view_id': day34_protocol.get('dataset_view_id'),
    'class_order': class_order,
    'k_values': K_VALUES,
    'arms': ARMS,
    'calibrator_specs': {
        key: calibrator.metadata()
        for key, calibrator in calibrator_registry.items()
    },
    'abstention_policies': abstention_policy_payload['policies'],
    'primary_operating_point': PRIMARY_OPERATING_POINT,
    'upstream': {
        'npz_sha256': NPZ_SHA256,
        'baseline_model_sha256': BASELINE_MODEL_SHA256,
        'day34_adapter_bundle_sha256': DAY34_ADAPTER_SHA256,
        'day34_gate_sha256': sha256_file(DAY34_GATE_PATH),
    },
    'governance': {
        'validation_fit_allowed': False,
        'validation_threshold_selection_allowed': False,
        'test_set_opened': False,
        'clinical_use_allowed': False,
    },
}
temporary_bundle = BUNDLE_JOBLIB.with_suffix(BUNDLE_JOBLIB.suffix + '.part')
joblib.dump(bundle_payload, temporary_bundle, compress=3)
temporary_bundle.replace(BUNDLE_JOBLIB)

artifact_paths = [
    INPUT_GATE_JSON,
    PARTITION_CSV,
    DEVELOPMENT_MANIFEST_CSV,
    DEVELOPMENT_PREDICTIONS_CSV_GZ,
    CALIBRATOR_CANDIDATES_CSV,
    CALIBRATOR_SELECTION_JSON,
    SCORE_SELECTION_CSV,
    DEVELOPMENT_CURVE_CSV,
    ABSTENTION_POLICY_JSON,
    VALIDATION_PREDICTIONS_CSV_GZ,
    VALIDATION_CALIBRATION_CSV,
    VALIDATION_SELECTIVE_CSV,
    VALIDATION_SUBJECT_CSV,
    VALIDATION_CLASS_CSV,
    VALIDATION_SESSION_CSV,
    VALIDATION_RELIABILITY_CSV,
    VALIDATION_CURVE_CSV,
    FIGURE_MANIFEST_JSON,
    PROTOCOL_JSON,
    BUNDLE_JOBLIB,
] + figure_paths

missing_artifacts = [str(path) for path in artifact_paths if not path.exists()]
if missing_artifacts:
    raise RuntimeError('Thiếu Day 35 artifacts:\n' + '\n'.join(missing_artifacts))

partition_sets = {
    partition: set(
        partition_frame.loc[
            partition_frame['day35_development_partition'] == partition,
            'subject_id',
        ].astype(str)
    )
    for partition in (
        'calibration_fit',
        'calibrator_select',
        'threshold_select',
    )
}
partition_overlap_count = sum(
    len(partition_sets[left] & partition_sets[right])
    for index, left in enumerate(partition_sets)
    for right in list(partition_sets)[index + 1:]
)
validation_development_overlap_count = len(
    set(validation_subjects)
    & set(partition_frame['subject_id'].astype(str))
)

checks = {
    'official_run_not_smoke_test': bool(not QUICK_SMOKE_TEST),
    'day34_input_gate_passed': bool(input_gate['status'] == 'PASS'),
    'day34_final_gate_passed': bool(
        str(day34_gate.get('status', '')).startswith('PASS')
    ),
    'baseline_model_not_refit': bool(BASELINE_REFIT_ALLOWED is False),
    'personalization_policy_not_refit': bool(
        PERSONALIZATION_POLICY_REFIT_ALLOWED is False
    ),
    'development_partitions_disjoint': bool(partition_overlap_count == 0),
    'validation_development_subjects_disjoint': bool(
        validation_development_overlap_count == 0
    ),
    'calibrator_fit_partition_only': True,
    'calibrator_selection_partition_only': True,
    'threshold_selection_partition_only': True,
    'validation_not_used_for_calibration_fit': bool(
        VALIDATION_CALIBRATOR_FIT_ALLOWED is False
    ),
    'validation_not_used_for_calibrator_selection': bool(
        VALIDATION_CALIBRATOR_SELECTION_ALLOWED is False
    ),
    'validation_not_used_for_threshold_selection': bool(
        VALIDATION_THRESHOLD_SELECTION_ALLOWED is False
    ),
    'calibration_preserved_argmax': bool(argmax_change_count == 0),
    'validation_scores_finite': bool(
        np.isfinite(
            validation_long_df[
                [
                    column
                    for column in validation_long_df.columns
                    if column.startswith('cal_prob__')
                ]
            ].to_numpy(dtype=float)
        ).all()
    ),
    'all_k_arm_policies_present': bool(
        set(abstention_policy_payload['policies'])
        == {f'k{k}:{arm}' for k in K_VALUES for arm in ARMS}
    ),
    'test_set_remained_closed': bool(TEST_SET_OPENED is False),
    'pooled_dataset_blocked': bool(POOLED_DATASET_ALLOWED is False),
    'clinical_use_blocked': bool(CLINICAL_USE_ALLOWED is False),
}
failed_checks = [
    name
    for name, value in checks.items()
    if not isinstance(value, (bool, np.bool_)) or not bool(value)
]
final_status = 'PASS' if not failed_checks else 'FAIL'

final_gate_payload = {
    'schema_version': 'day35-final-gate.v2',
    'created_at_utc': utc_now_iso(),
    'status': final_status,
    'dataset_profile': DATASET_PROFILE,
    'checks': {name: bool(value) for name, value in checks.items()},
    'counts': {
        'training_subjects': int(len(train_subjects)),
        'validation_subjects': int(len(validation_subjects)),
        'development_prediction_rows': int(len(development_predictions_df)),
        'validation_prediction_rows_long': int(len(validation_long_df)),
        'development_partition_overlap_count': int(partition_overlap_count),
        'validation_development_overlap_count': int(
            validation_development_overlap_count
        ),
        'calibration_argmax_change_count': int(argmax_change_count),
        'calibrator_count': int(len(calibrator_registry)),
        'abstention_policy_count': int(
            len(abstention_policy_payload['policies'])
        ),
    },
    'failed_checks': failed_checks,
    'performance_is_not_a_structural_gate': True,
}
write_json_atomic(final_gate_payload, FINAL_GATE_JSON)

artifact_hashes = {
    path.name: {
        'sha256': sha256_file(path),
        'size_bytes': int(path.stat().st_size),
        'runtime_path': str(path),
        'persistent_path': str(PERSIST_ROOT / path.name),
    }
    for path in artifact_paths + [FINAL_GATE_JSON]
}

primary_summary = validation_selective_metrics_df[
    (validation_selective_metrics_df['scope'] == PROFILE['primary_scope'])
    & (
        validation_selective_metrics_df['operating_point']
        == PRIMARY_OPERATING_POINT
    )
].to_dict(orient='records')

evidence_payload = {
    'schema_version': 'day35-confidence-calibration-evidence.v2',
    'created_at_utc': utc_now_iso(),
    'status': final_status,
    'run_id': RUN_ID,
    'scope': 'confidence_calibration_and_selective_prediction',
    'input': input_gate,
    'protocol': protocol_payload,
    'calibrator_selection': calibrator_selection_payload,
    'abstention_policy': abstention_policy_payload,
    'primary_validation_summary': primary_summary,
    'governance': {
        'baseline_refit_allowed': BASELINE_REFIT_ALLOWED,
        'personalization_policy_refit_allowed': (
            PERSONALIZATION_POLICY_REFIT_ALLOWED
        ),
        'validation_calibrator_fit_allowed': (
            VALIDATION_CALIBRATOR_FIT_ALLOWED
        ),
        'validation_calibrator_selection_allowed': (
            VALIDATION_CALIBRATOR_SELECTION_ALLOWED
        ),
        'validation_threshold_selection_allowed': (
            VALIDATION_THRESHOLD_SELECTION_ALLOWED
        ),
        'test_set_opened': TEST_SET_OPENED,
        'pooled_dataset_allowed': POOLED_DATASET_ALLOWED,
        'clinical_use_allowed': CLINICAL_USE_ALLOWED,
    },
    'interpretation_limits': [
        'Abstention means insufficient confidence for automated routing; it is not a new gesture class.',
        'Coverage-risk results are offline and dataset-specific.',
        'No clinical safety claim follows from Mendeley or GRABMyo performance.',
        'Repeated few-shot episodes are summarized with subject-level uncertainty; row count is not an independent sample count.',
    ],
    'technical_debt': [
        'External-device and real Noraxon export calibration remain unvalidated.',
        'Operating points require stakeholder risk tolerance before any pilot.',
        'Day 36 should consume calibrated confidence and abstention as context, not hard diagnosis.',
    ],
    'artifacts': artifact_hashes,
    'next_step': (
        'Day 36 fatigue-context architecture using calibrated confidence, '
        'coverage-risk policy and abstention state.'
    ),
}
write_json_atomic(evidence_payload, EVIDENCE_JSON)

for path in (PROTOCOL_JSON, BUNDLE_JOBLIB, EVIDENCE_JSON, FINAL_GATE_JSON):
    persist_artifact(path)

zip_members = artifact_paths + [EVIDENCE_JSON, FINAL_GATE_JSON]
temporary_zip = HANDOFF_ZIP.with_suffix(HANDOFF_ZIP.suffix + '.part')
temporary_zip.unlink(missing_ok=True)
with zipfile.ZipFile(
    temporary_zip,
    mode='w',
    compression=zipfile.ZIP_DEFLATED,
) as archive:
    for path in zip_members:
        archive.write(path, arcname=path.name)
temporary_zip.replace(HANDOFF_ZIP)
persist_artifact(HANDOFF_ZIP)

if failed_checks:
    raise RuntimeError(
        'Day 35 final gate failed: '
        + json.dumps(failed_checks, ensure_ascii=False)
    )

print('=' * 96)
print('[CELL 10] DAY 35 FINAL SUMMARY')
print('=' * 96)
print(f'Status                         : {final_status}')
print(f'Dataset profile                : {DATASET_PROFILE}')
print(f'Train subjects                 : {len(train_subjects)}')
print(f'Validation subjects            : {len(validation_subjects)}')
print(f'Calibrators                    : {len(calibrator_registry)}')
print(f'Abstention policies            : {len(abstention_policy_payload["policies"])}')
print(f'Bundle                         : {PERSIST_ROOT / BUNDLE_JOBLIB.name}')
print(f'Evidence                       : {PERSIST_ROOT / EVIDENCE_JSON.name}')
print(f'Final gate                     : {PERSIST_ROOT / FINAL_GATE_JSON.name}')
print(f'Handoff ZIP                    : {PERSIST_ROOT / HANDOFF_ZIP.name}')
print('[PASS] Day 35 confidence calibration and abstention completed.')

if DOWNLOAD_HANDOFF_TO_BROWSER and IN_COLAB:
    files.download(str(PERSIST_ROOT / HANDOFF_ZIP.name))

[CELL 10] DAY 35 FINAL SUMMARY
Status                         : PASS
Dataset profile                : grabmyo
Train subjects                 : 34
Validation subjects            : 9
Calibrators                    : 4
Abstention policies            : 4
Bundle                         : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day35-confidence/day35-confidence-grabmyo-v2/day35-calibration-abstention-bundle.joblib
Evidence                       : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day35-confidence/day35-confidence-grabmyo-v2/day35-confidence-calibration-evidence.json
Final gate                     : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day35-confidence/day35-confidence-grabmyo-v2/day35-final-gate.json
Handoff ZIP                    : /content/drive/MyDrive/MyoLab-AI-data/grabmyo-physionet-v1.1.0/outputs/day35-confidence/day35-confidence-grabmyo-v2/day35-confidence-calibration-handoff.zip
[PAS

## Artifacts cần giữ sau Day 35

File bàn giao chính:

```text
day35-confidence-calibration-handoff.zip
```

Bundle dùng cho Day 36:

```text
day35-calibration-abstention-bundle.joblib
```

Các bảng quan trọng:

```text
day35-validation-calibration-metrics.csv
day35-validation-selective-metrics.csv
day35-validation-subject-metrics.csv
day35-validation-class-metrics.csv
day35-validation-session-metrics.csv
day35-validation-predictions.csv.gz
day35-abstention-policy.json
day35-final-gate.json
```

Chạy riêng từng dataset:

```python
import os
os.environ["DAY35_DATASET_PROFILE"] = "mendeley"
```

hoặc:

```python
import os
os.environ["DAY35_DATASET_PROFILE"] = "grabmyo"
```